# Projeto de grupo: as notas dos filmes dependem de quem as dá?
### IMDb + TMDB + MovieLens num Data Lakehouse (arquitetura Medallion), seguindo o CRISP-DM

**Elementos:** Francisco Monteiro (nº 20251694) · José Grave (nº 20251843) · Luís Pousada (nº 20250979) · Tiago Felgueira (nº 20242047)
**Unidade curricular:** Analysing Big Data 2026/2027, NOVA IMS 

### Como este notebook está organizado

Usamos duas ideias que se complementam:

* a **Arquitetura Medallion** organiza os **dados**: bronze = bruto com histórico, silver = limpo e integrado, gold = pronto para análise;
* o **CRISP-DM** organiza o **processo de análise**: perceber o problema, perceber os dados, preparar, modelar, avaliar e pôr em produção.

O notebook segue o percurso dos dados de cima para baixo, como um pipeline: cada etapa só usa o que as etapas anteriores produziram.

| Etapa | Fase CRISP-DM | Camada Medallion | O que faz |
|---|---|---|---|
| 0 | — | — | Parâmetros, constantes e criação dos schemas e do Volume |
| 1 | Business Understanding | — | Pergunta de negócio, hipóteses e arquitetura |
| 2 | Data Understanding | Volume `raw` + **Bronze** | Obter e carregar IMDb/MovieLens com Auto Loader; dados JSON complementares por upload manual |
| 3 | Data Preparation | **Silver** | Limpar, tipar, deduplicar e integrar o IMDb e o MovieLens |
| 4 | Modeling | **Gold** | Uma tabela por hipótese, regressão (Spark ML), perguntas descritivas e star schema |
| 5 | Evaluation | Gold + gráficos | Figuras, evidências técnicas e organização física |
| 6 | Deployment | — | Job, consultas SQL e dashboard, inventário final |
| 7 | — | — | Conclusões |

### Como executar

1. Importar o notebook e escolher **Serverless** como compute.
2. O widget `movielens` vem com **`32m`** (~32 milhões de avaliações), a versão da execução final. Para testes rápidos, escolher **`small`** (~100 mil avaliações, corre em poucos minutos). Ao mudar de versão, correr primeiro o notebook `00_reset`.
3. Fazer **Run all**. O widget `catalogo` vem com `workspace`, o catálogo que existe em todos os workspaces da Free Edition, por isso não é preciso criar nada antes.
4. Os ficheiros do MovieLens e do IMDb são descarregados automaticamente, se a conta tiver acesso à internet; se não, carregam-se à mão (ver secção 2).

### Análises opcionais

O notebook preserva as análises de idioma, orçamento e retorno financeiro, bem como as Figuras 6 e 9. Para as executar, carregar os três ficheiros JSONL no Volume `raw/tmdb`, descomentar as linhas marcadas `[TMDB]` nas células correspondentes e voltar a executar o notebook. O projeto mantém 17 figuras numeradas; as Figuras 6 e 9 ficam disponíveis quando estes módulos estão ativos.
Requer a introdução do dataset tmdb manualmente que não se conseguiu introduzir nesta edição do Databricks (Free edition)
### Convenção de nomes

* **Funções, variáveis e comentários em português** (`gravar_delta`, `mais_recente`, `populacoes_analise`).
* **Colunas e tabelas em inglês**: as colunas vêm assim das fontes (`movieId`, `tconst`, `vote_average`) e mantivemos o mesmo idioma nas tabelas e colunas que criamos, para que as consultas SQL leiam de forma uniforme.

**Padrões de PySpark que aparecem muitas vezes:**
* `F.col("x")` refere uma coluna; `withColumn` cria ou substitui uma coluna; `groupBy(...).agg(...)` agrega; `join` junta tabelas.
* `Window.partitionBy(...).orderBy(...)` faz contas "por grupo sem agrupar" (ex.: ficar com a linha mais recente de cada filme).
* Nada é calculado até haver uma ação (`count`, `write`, `display`): o Spark é *lazy* e só executa nesse momento.

## 0. Setup

Esta célula prepara tudo o que as etapas seguintes usam: bibliotecas, parâmetros (widgets), constantes e a estrutura no Unity Catalog (schemas `bronze`, `silver`, `gold` e o Volume `raw`).

**Porque mudámos para o catálogo por omissão `workspace`.** A abordagem anterior não estava a correr como precisávamos: usávamos um catálogo próprio, `abd_project`, criado à mão por um elemento do grupo, e o notebook só corria por inteiro nessa conta. Criar o catálogo dentro do notebook também não resolvia o problema, porque o dono passava a ser quem o corresse primeiro. Por isso mudámos para `workspace`, que existe em todos os workspaces da Databricks Free Edition e onde o utilizador tem permissões. Assim, o notebook pode correr noutra conta, incluindo a do professor, sem preparação manual. Os schemas e o Volume são criados com `IF NOT EXISTS`: se já existirem, são usados como estão.

In [0]:
# =============================================================================
# ETAPA 0 — SETUP
# Bibliotecas, parâmetros, constantes e estrutura no Unity Catalog.
# =============================================================================
import fnmatch
import gzip
import json
import os
import re
import shutil
import urllib.request
import uuid
import zipfile

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
import numpy as np
import pandas as pd
from pyspark.sql import Window
from pyspark.sql import functions as F

# -----------------------------------------------------------------------------
# 0.1 Parâmetros (widgets que aparecem no topo do notebook)
# -----------------------------------------------------------------------------
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
dbutils.widgets.dropdown("movielens", "32m", ["32m", "small"], "Versão do MovieLens")
dbutils.widgets.text("min_votos_imdb", "1000", "Mín. votos IMDb")
dbutils.widgets.text("min_avaliacoes_ml", "10", "Mín. avaliações MovieLens")
# [TMDB desativada] 
# dbutils.widgets.text("min_votos_tmdb", "20", "Mín. votos TMDB")
dbutils.widgets.dropdown("publicar_sql", "sim", ["sim", "nao"], "Publicar consultas e dashboard")

CATALOGO          = dbutils.widgets.get("catalogo").strip()
VERSAO_MOVIELENS  = dbutils.widgets.get("movielens")                # 32m (por omissão, execução final) ou small (testes rápidos)
MIN_VOTOS_IMDB    = int(dbutils.widgets.get("min_votos_imdb"))      # filmes com menos votos são ruído
MIN_AVALIACOES_ML = int(dbutils.widgets.get("min_avaliacoes_ml"))
# [TMDB desativada] 
# MIN_VOTOS_TMDB    = int(dbutils.widgets.get("min_votos_tmdb"))
PUBLICAR_SQL      = dbutils.widgets.get("publicar_sql") == "sim"     # usado na secção 6.1

# -----------------------------------------------------------------------------
# 0.2 Constantes: nomes das camadas, do Volume e das tabelas, num só sítio
# -----------------------------------------------------------------------------
BRONZE, SILVER, GOLD = "bronze", "silver", "gold"
CAMADAS = [BRONZE, SILVER, GOLD]
VOLUME = "raw"
RAIZ_RAW = f"/Volumes/{CATALOGO}/{BRONZE}/{VOLUME}"     # pasta dos ficheiros originais
PASTA_CHECKPOINTS = f"{RAIZ_RAW}/_checkpoints"          # memória do Auto Loader (secção 2)

# Cada versão do MovieLens tem a sua pasta no Volume, para os ficheiros das duas nunca se misturarem
PASTAS_FONTES = {"imdb": "imdb", "movielens": "movielens" if VERSAO_MOVIELENS == "small" else "movielens_32m"}


def pasta_fonte(fonte):
    """Caminho da pasta da fonte no Volume (o MovieLens depende da versão escolhida no widget)."""
    return f"{RAIZ_RAW}/{PASTAS_FONTES[fonte]}"
FONTES = ("imdb", "movielens")   # [TMDB desativada] "tmdb"

# Identificador desta execução: todas as linhas que entram na bronze ficam marcadas com ele
LOTE = spark.sql("SELECT date_format(current_timestamp(), 'yyyyMMdd_HHmmss')").first()[0]

# Tabelas de cada camada (usadas no diagrama do pipeline e na secção 5.11)
TABELAS_BRONZE = {
                  # [TMDB desativada] "tmdb": ["tmdb_movie_lists", "tmdb_genres", "tmdb_movie_details"],
                  "imdb": ["imdb_title_basics", "imdb_title_ratings"],
                  "movielens": ["ml_ratings", "ml_movies", "ml_links", "ml_tags"]}
TABELAS_SILVER = {
                  # [TMDB desativada] "tmdb": ["tmdb_movies", "tmdb_movie_details", "tmdb_movie_genres"],
                  "imdb": ["imdb_movies", "imdb_movie_genres"],
                  "movielens": ["ml_ratings", "ml_movies", "ml_links", "ml_tags"]}
TABELAS_GOLD = ["ratings_by_decade", "gap_by_genre",
                # [TMDB desativada] "gap_by_language", "gap_by_budget", "roi_by_genre",
                "movie_rating_gaps", "tags_by_gap", "gap_model_coefficients", "dq_results",
                # perguntas descritivas e de benchmarking (secção 4.3)
                "genre_catalog", "user_activity", "top_movies", "ratings_over_time", "flag_sensitivity",
                "movie_rating_shifts", "movie_day_anomalies", "anomaly_participants"]
# Modelo dimensional (star schema) na gold, secção 4.4
TABELAS_STAR = ["fact_rating", "bridge_movie_genre", "dim_movie", "dim_source", "dim_year", "dim_genre"]   # [TMDB desativada] "dim_language"

# [TMDB desativada] 
# # Nomes dos idiomas (códigos ISO do TMDB), usados nos gráficos
# NOMES_IDIOMAS = {"en": "Inglês", "fr": "Francês", "it": "Italiano", "ja": "Japonês", "es": "Espanhol",
                 # "de": "Alemão", "ko": "Coreano", "zh": "Mandarim", "cn": "Cantonês", "hi": "Hindi",
                 # "sv": "Sueco", "ru": "Russo", "da": "Dinamarquês", "pt": "Português", "nl": "Neerlandês"}

# Estilo comum a todos os gráficos: uma cor fixa por fonte
CORES_FONTES = {"IMDb": "#1f77b4", "MovieLens": "#ff7f0e"}   # [TMDB desativada] "TMDB": "#2ca02c"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})


def nome_tabela(camada, tabela):
    """Nome completo de uma tabela no Unity Catalog: catálogo.camada.tabela."""
    return f"{CATALOGO}.{camada}.{tabela}"


def definir_descricao(nome_completo, texto):
    """Guarda a descrição de uma tabela no Unity Catalog (aparece no Catalog Explorer)."""
    texto = texto.replace("'", "''")   # em SQL, uma aspa dentro de um texto escreve-se duas vezes
    spark.sql(f"COMMENT ON TABLE {nome_completo} IS '{texto}'")


# -----------------------------------------------------------------------------
# 0.3 Estrutura no Unity Catalog: catálogo > schemas > Volume > pastas das fontes
# -----------------------------------------------------------------------------
catalogos = [linha[0] for linha in spark.sql("SHOW CATALOGS").collect()]
if CATALOGO not in catalogos:
    raise ValueError(f"O catálogo '{CATALOGO}' não existe. Catálogos disponíveis: {catalogos}")

for camada in CAMADAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{camada}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{BRONZE}.{VOLUME}")
for fonte in FONTES:
    dbutils.fs.mkdirs(pasta_fonte(fonte))   # não faz nada se a pasta já existir

# Datas e horas sempre em UTC, para os resultados não dependerem do fuso de quem corre
spark.conf.set("spark.sql.session.timeZone", "UTC")

print(f"Setup concluído | catálogo = {CATALOGO} | Volume = {RAIZ_RAW} | MovieLens = {VERSAO_MOVIELENS} | lote = {LOTE}")

## 1. Business Understanding (CRISP-DM fase 1)

**Contexto.** Quem escolhe um filme olha para uma nota. Mas há várias: o IMDb (votos de qualquer pessoa registada), o TMDB (comunidade do The Movie Database) - não incorporado na análise que se segue - e o MovieLens (utilizadores de um sistema de recomendação académico). Uma plataforma de streaming que queira mostrar "a nota" de um filme precisa de saber se estas fontes concordam e quando não concordam.

**Pergunta de negócio.** As notas dos filmes diferem entre fontes? Que fatores (género, década, idioma, orçamento) explicam essas diferenças?

**Hipóteses a testar**
* **H1:** as fontes dão notas médias diferentes ao mesmo conjunto de filmes.
* **H2:** a diferença IMDb − MovieLens depende do género.
* **H3:** filmes em idioma não inglês têm diferenças maiores.
* **H4:** filmes de orçamento alto têm notas mais consensuais.
* **H5:** os filmes antigos têm notas mais altas (efeito de sobrevivência: só os clássicos continuam a ser vistos).

**Critério de sucesso.** Responder a cada hipótese com uma tabela gold e um gráfico, e quantificar o efeito de cada fator num modelo de regressão.

**Dados.** Estruturados: notas, votos, duração e datas das avaliações. Não estruturados: tags de texto livre do MovieLens. Streaming: a ingestão da bronze do IMDb e do MovieLens é feita com Auto Loader, que trata os ficheiros como eventos que vão chegando.

**Chave de integração.** O `links.csv` do MovieLens liga `movieId` ↔ `imdbId` ↔ `tmdbId`. O IMDb usa `tconst` (`tt0114709`), construído a partir do `imdbId`.

Nota: Para a integração do tmdb é preciso descomentar secções de código.

### 1.1 Arquitetura da solução (Lakehouse + Medallion)

| Camada | Onde fica | O que contém | Regra principal |
|---|---|---|---|
| **Fontes** | Volume `bronze.raw` | IMDb (TSV), MovieLens (CSV) e TMDB (JSON, upload manual) | download automático ou upload manual; ficheiros nunca alterados |
| **Bronze** | tabelas `bronze.*` | dados brutos em texto + auditoria (`_batch_id`, `_source_file`, `_ingested_at`) | só entram ficheiros novos, sempre em **append** (Auto Loader) |
| **Silver** | tabelas `silver.*` | dados limpos, tipados, sem duplicados; `movies_unified` integra o IMDb e o MovieLens | versão mais recente de cada chave; reconstruível a partir da bronze |
| **Gold** | tabelas `gold.*` | agregados por década e género; modelo; qualidade; star schema | uma tabela por hipótese, pronta para gráficos; star schema ao detalhe |
| **Consumo** | este notebook, consultas SQL e dashboard | 17 figuras, regressão, conclusões | lê só a gold |

In [0]:
# =============================================================================
# ETAPA 1.1 — Diagrama da arquitetura
# Desenho em HTML/CSS (cartões lado a lado). Fica visível no notebook e no HTML exportado.
# =============================================================================
def mostrar_arquitetura():
    """Desenha as cinco camadas da arquitetura como cartões coloridos, ligados por setas."""
    camadas = [
        ("Fontes", "#e8eef7", "#4c72b0", "Ficheiros originais no Volume", [
            ("IMDb", "title.basics, title.ratings (TSV, download automático)"),
            ("MovieLens", "ratings, movies, links, tags (CSV, download automático)")]),
            # [TMDB desativada] 
            # ("TMDB", "listas, géneros, detalhe (JSON, upload manual)")]),
        ("Bronze", "#f6e7d8", "#a0522d", "Dados brutos com histórico de leituras", [
            ("Auto Loader", "lê só os ficheiros novos (streaming incremental)"),
            ("6 tabelas Delta", "tudo em texto + _batch_id, _source_file, _ingested_at"),
            ("Regra", "só append")]),
        ("Silver", "#eef0f2", "#6c757d", "Dados limpos e integrados", [
            ("Por fonte", "imdb_movies, ml_ratings, ml_movies, ml_links, ml_tags…"),
            ("Integrada", "movies_unified (IMDb ⨝ MovieLens)"),
            ("Qualidade", "chaves únicas, tipos, regras em dq_results")]),
        ("Gold", "#fbf3d5", "#b8860b", "Pronta para responder às hipóteses", [
            ("Comparação", "ratings_by_decade, gap_by_genre"),
            # [TMDB desativada] 
            # ("Fatores", "gap_by_language, gap_by_budget, roi_by_genre"),
            ("Modelo e texto", "gap_model_coefficients, tags_by_gap"),
            ("Star schema", "fact_rating + 4 dimensões + ponte de géneros")]),
        ("Consumo", "#e6f4ea", "#2e7d32", "Análise e decisão", [
            ("Gráficos", "figuras com intervalos de confiança"),
            ("Modelo", "regressão do gap IMDb − MovieLens (Spark ML)"),
            ("SQL", "consultas guardadas e dashboard")]),
    ]
    cartoes = []
    for i, (nome, fundo, cor, subtitulo, itens) in enumerate(camadas):
        linhas = "".join(f'<div style="margin:6px 0"><b>{k}</b><br><span style="color:#555;overflow-wrap:anywhere">{v}</span></div>'
                         for k, v in itens)
        cartoes.append(
            f'<div style="flex:1 1 150px;min-width:150px;background:{fundo};border-top:6px solid {cor};'
            f'border-radius:10px;padding:10px 12px;box-shadow:0 1px 3px rgba(0,0,0,.12)">'
            f'<div style="font-size:18px;font-weight:700;color:{cor}">{nome}</div>'
            f'<div style="font-size:12px;font-style:italic;color:#333;margin-bottom:6px">{subtitulo}</div>'
            f'<div style="font-size:12px;color:#222">{linhas}</div></div>')
        if i < len(camadas) - 1:
            cartoes.append('<div style="align-self:center;font-size:22px;color:#888;padding:0">➜</div>')
    displayHTML(f"""
    <div style="background:#ffffff;color:#222;font-family:Segoe UI,Helvetica,Arial,sans-serif;padding:12px">
      <div style="font-size:20px;font-weight:700;margin-bottom:2px">Arquitetura Lakehouse · Medallion</div>
      <div style="font-size:13px;color:#555;margin-bottom:12px">
        Unity Catalog: <code>{CATALOGO}.bronze</code> · <code>.silver</code> · <code>.gold</code>
        &nbsp;|&nbsp; Delta Lake &nbsp;|&nbsp; Spark (serverless)</div>
      <div style="display:flex;flex-wrap:wrap;gap:8px;align-items:stretch">{''.join(cartoes)}</div>
      <div style="display:flex;gap:16px;flex-wrap:wrap;font-size:12px;color:#444;margin-top:12px">
        <span><b>Incremental:</b> ficheiros já lidos não voltam a entrar</span>
        <span><b>Sem perda de dados:</b> bronze só em append; histórico Delta e <code>RESTORE</code></span>
        <span><b>Orquestração:</b> Job descrito na secção 6</span>
      </div>
    </div>""")


mostrar_arquitetura()

## 2. Data Understanding: ingestão para a Bronze (CRISP-DM fase 2)

**IMDb e MovieLens** são as fontes carregadas automaticamente no percurso principal. O notebook também preserva as células da TMDB, que usam os ficheiros JSONL carregados manualmente e leitura batch; estas células estão assinaladas com `[TMDB]`.

Cada fonte passa por dois movimentos diferentes, que convém não confundir:

1. **Fonte → Volume `raw`**: pôr os ficheiros dentro do Databricks.
2. **Volume `raw` → tabelas bronze**: ler esses ficheiros para tabelas Delta, só os que ainda não foram lidos.

| Fonte | 1. Como chega ao Volume | 2. Como passa do Volume para a bronze |
|---|---|---|
| **MovieLens** | download automático do zip da GroupLens, na versão escolhida no widget (`small` em `raw/movielens`, `32m` em `raw/movielens_32m`); se falhar, upload manual | **Auto Loader** (Structured Streaming) |
| **IMDb** | download automático dos ficheiros oficiais; se falhar, upload manual | **Auto Loader** (Structured Streaming) |
| **TMDB** (upload manual) | upload dos 3 ficheiros JSONL para o Volume | leitura batch incremental |

**Planos para pôr os ficheiros no Volume, por ordem:**

| Plano | O que faz | Quando entra |
|---|---|---|
| A | usa os ficheiros que já estão no Volume | sempre primeiro |
| B | download dos sites oficiais (GroupLens e IMDb) | se faltarem ficheiros |
| B2 | download de mirrors públicos na Kaggle, com verificação das colunas | se a rede da conta bloquear os sites oficiais |
| C | pára e diz que ficheiros carregar à mão e em que pasta | se nada disto resultar |

**Porque existe o plano B2.** Na Free Edition, o compute só chega a uma lista de domínios autorizados, que não é pública e varia entre contas: no nosso grupo, numa conta os sites oficiais funcionavam e noutra estavam bloqueados, mas a Kaggle passava. Os mirrors são **cópias de terceiros**: o do MovieLens 32M é a edição oficial de 2023-10; o do IMDb é um dump de outubro de 2023, mais antigo do que o ficheiro oficial atual. Por isso, **as notas e os votos do IMDb, e os resultados que dependem deles, podem diferir entre uma execução com o plano B e outra com o B2.** A execução final do trabalho deve usar sempre o mesmo plano. Antes de ir para o Volume, cada ficheiro é confirmado pelo cabeçalho (nomes das colunas). Há mirror para as duas versões do MovieLens (`32m` e `small`).

**Porque é que o download é automático com plano B manual.** O objetivo é que o notebook corra noutra conta sem preparação, por exemplo na do professor. O MovieLens e o IMDb publicam ficheiros abertos, por isso o próprio notebook os pode descarregar. Há dois limites:
* **Ficheiros JSON da TMDB:** carregar os três JSONL em *Catalog > (catálogo) > bronze > raw > tmdb > Upload to this volume* antes de ativar as células correspondentes.
* **A Free Edition pode bloquear o acesso à internet**, dependendo da conta. Nesse caso o download falha e entra o plano B: carregar os ficheiros à mão para a pasta indicada na mensagem.

**Duas versões do MovieLens.** O widget `movielens` (por omissão `32m`) escolhe entre o `small` (~100 mil avaliações, 610 utilizadores, ~9.700 filmes) e o `32m` (~32 milhões de avaliações, ~200 mil utilizadores, ~87 mil filmes). Os ficheiros das duas têm os mesmos nomes e colunas, por isso o resto do notebook não muda. Cada versão fica na sua pasta do Volume, e o notebook pára com uma mensagem se a bronze já tiver dados da outra versão: misturá-las juntaria utilizadores e avaliações de amostras diferentes.

O download é feito primeiro para uma pasta temporária (`/tmp`) e só depois copiado para o Volume. Assim, um download interrompido nunca deixa no Volume um ficheiro incompleto, que seria lido como se fosse válido.

### Como funciona o Auto Loader (IMDb e MovieLens)

O Auto Loader (`cloudFiles`, aula 8) é a ferramenta do Databricks para ingestão incremental de ficheiros. Funciona sobre Structured Streaming: cada ficheiro novo na pasta é tratado como um evento que chega.

```
 pasta raw/movielens ──► Auto Loader ──► compara com o checkpoint ──► lê só os novos ──► append em bronze.ml_ratings
                         (readStream)    (_checkpoints/ml_ratings:      (schema em texto     e regista-os no checkpoint
                                          ficheiros já lidos)            + auditoria)
```

Em cada execução:
1. `spark.readStream.format("cloudFiles")` aponta para a pasta da fonte; `pathGlobFilter` escolhe os ficheiros de cada tabela (ex.: `ratings*.csv`).
2. O Auto Loader lista a pasta e compara-a com o ***checkpoint***, uma pasta no Volume onde guarda os ficheiros que já leu.
3. Lê só os ficheiros novos, acrescenta as colunas de auditoria e escreve-os na tabela bronze em **append**.
4. Regista esses ficheiros no checkpoint. Se a execução falhar a meio, recomeça de onde parou e nenhum ficheiro entra duas vezes (garantia *exactly-once*).
5. Com `trigger(availableNow=True)`, processa tudo o que há de novo e **pára**, como uma leitura normal. É o modo suportado no compute serverless e o que permite correr o notebook como um Job.

### Porque é que os ficheiros JSON da TMDB são lidos em batch

O Auto Loader também lê JSON, pelo que ambos os métodos são possíveis. Para estes ficheiros escolhemos a leitura batch por três razões:
* **Leitura em batch:** os JSONL são carregados como uma extração fechada; a tabela bronze regista os ficheiros por caminho e data de modificação para aceitar novas adições sem reler os anteriores.
* **A leitura batch dá o mesmo resultado**, com o controlo feito de forma explícita: lê os ficheiros da pasta, compara cada um (caminho e data de modificação) com os que já estão na bronze e acrescenta só os novos (`left_anti` join). Não há checkpoint: a própria tabela bronze serve de memória.
* **Mostra duas formas de ingestão incremental lado a lado:** streaming com Auto Loader (IMDb e MovieLens) e batch com controlo próprio (TMDB). A alternativa com Auto Loader está preservada na célula 2.1.

| | Auto Loader (IMDb, MovieLens) | Batch incremental (TMDB) |
|---|---|---|
| Leitura | `spark.readStream.format("cloudFiles")` | `spark.read.format("json")` |
| Memória dos ficheiros lidos | checkpoint em `raw/_checkpoints/<tabela>` | colunas `_source_file` e `_file_modified` da própria tabela |
| Como evita duplicados | o checkpoint ignora ficheiros já lidos | `left_anti` join com os ficheiros já na bronze |
| Ficheiro com o mesmo nome, mas alterado | ignorado (por omissão) | lido outra vez (a data de modificação mudou) |
| Escrita | `writeStream ... toTable`, append | `write ... saveAsTable`, append |

**Simular uma segunda leitura da fonte:** carregar um ficheiro novo com o mesmo prefixo (ex.: `movie_details_2026-11-10.jsonl`) e voltar a correr. Só esse ficheiro entra, nos dois métodos.

Na bronze todas as colunas ficam como **texto**, tal como vêm da fonte; os tipos só são definidos na silver. Cada linha recebe colunas de auditoria: fonte, ficheiro de origem, data do ficheiro, lote e momento da ingestão.

In [0]:
# =============================================================================
# ETAPA 2.0 — Funções da ingestão: obter os ficheiros e carregá-los com Auto Loader
# =============================================================================

# -----------------------------------------------------------------------------
# O que se lê de cada fonte: (padrão do nome do ficheiro, tabela bronze, formato,
# schema, opções de leitura, descrição da tabela).
# O padrão aceita versões novas do mesmo ficheiro, ex.: movie_details_2026-11-10.jsonl.
# -----------------------------------------------------------------------------
def schema_texto(colunas):
    """Schema em que todas as colunas são texto: a bronze guarda os valores tal como vêm."""
    return ", ".join(f"`{c}` STRING" for c in colunas)


# IMDb: TSV com separador tab e sem aspas (os títulos podem conter aspas)
OPCOES_IMDB = {"header": "true", "sep": "\t", "quote": ""}
# MovieLens: CSV com aspas duplicadas dentro dos títulos e tags (daí escape='"');
# movies e tags podem ter texto com quebras de linha (daí multiLine)
OPCOES_ML = {"header": "true", "escape": '"'}
OPCOES_ML_TEXTO = {"header": "true", "escape": '"', "multiLine": "true"}

INGESTOES = {
    # [TMDB desativada] 
    # "tmdb": [
        # # o 'payload' guarda o JSON completo de cada filme como texto (componente não estruturada)
        # ("movie_lists*.jsonl", "tmdb_movie_lists", "json", "list_name STRING, page INT, payload STRING", {},
         # "TMDB: listas popular, top_rated e now_playing (JSON bruto, histórico de leituras)"),
        # ("genres*.jsonl", "tmdb_genres", "json", "payload STRING", {},
         # "TMDB: lista de géneros (JSON bruto)"),
        # ("movie_details*.jsonl", "tmdb_movie_details", "json", "movie_id BIGINT, payload STRING", {},
         # "TMDB: detalhe por filme (JSON bruto, histórico de leituras)"),
    # ],
    "imdb": [
        ("title.basics*.tsv.gz", "imdb_title_basics", "csv",
         schema_texto(["tconst", "titleType", "primaryTitle", "originalTitle", "isAdult",
                       "startYear", "endYear", "runtimeMinutes", "genres"]), OPCOES_IMDB,
         "IMDb title.basics, valores brutos em texto"),
        ("title.ratings*.tsv.gz", "imdb_title_ratings", "csv",
         schema_texto(["tconst", "averageRating", "numVotes"]), OPCOES_IMDB,
         "IMDb title.ratings, valores brutos em texto"),
    ],
    "movielens": [
        ("ratings*.csv", "ml_ratings", "csv", schema_texto(["userId", "movieId", "rating", "timestamp"]), OPCOES_ML,
         "MovieLens ratings.csv, valores brutos em texto"),
        ("movies*.csv", "ml_movies", "csv", schema_texto(["movieId", "title", "genres"]), OPCOES_ML_TEXTO,
         "MovieLens movies.csv, valores brutos em texto"),
        ("links*.csv", "ml_links", "csv", schema_texto(["movieId", "imdbId", "tmdbId"]), OPCOES_ML,
         "MovieLens links.csv, valores brutos em texto"),
        ("tags*.csv", "ml_tags", "csv", schema_texto(["userId", "movieId", "tag", "timestamp"]), OPCOES_ML_TEXTO,
         "MovieLens tags.csv, valores brutos em texto"),
    ],
}

# -----------------------------------------------------------------------------
# Passo 1: garantir que os ficheiros estão no Volume (plano A: já lá estão;
# plano B: download automático; plano C: upload manual, com instruções)
# -----------------------------------------------------------------------------
# Versões do MovieLens: (endereço do zip, pasta dentro do zip). A escolha vem do widget "movielens".
VERSOES_MOVIELENS = {
    "small": ("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", "ml-latest-small"),  # ~100 mil avaliações
    "32m":   ("https://files.grouplens.org/datasets/movielens/ml-32m.zip", "ml-32m"),                     # ~32 milhões de avaliações
}
URL_MOVIELENS, PASTA_ZIP_MOVIELENS = VERSOES_MOVIELENS[VERSAO_MOVIELENS]
URLS_IMDB = {"title.basics.tsv.gz": "https://datasets.imdbws.com/title.basics.tsv.gz",
             "title.ratings.tsv.gz": "https://datasets.imdbws.com/title.ratings.tsv.gz"}


def ficheiros_na_pasta(fonte, padrao):
    """Nomes dos ficheiros da pasta da fonte que correspondem ao padrão (lista vazia se não houver)."""
    return [f.name for f in dbutils.fs.ls(pasta_fonte(fonte)) if fnmatch.fnmatch(f.name, padrao)]


def descarregar(url, destino):
    """Descarrega um ficheiro para /tmp e só depois o copia para o Volume.
    Se o download falhar a meio, o Volume fica sem ficheiro em vez de ficar com um ficheiro incompleto."""
    temporario = f"/tmp/{os.path.basename(destino)}"
    with urllib.request.urlopen(url, timeout=120) as resposta, open(temporario, "wb") as ficheiro:
        shutil.copyfileobj(resposta, ficheiro)
    shutil.copy(temporario, destino)


def descarregar_movielens(pasta):
    """MovieLens: descarrega o zip da GroupLens (versão do widget) e copia os 4 CSV para a pasta da fonte."""
    zip_tmp = f"/tmp/{PASTA_ZIP_MOVIELENS}.zip"
    with urllib.request.urlopen(URL_MOVIELENS, timeout=120) as resposta, open(zip_tmp, "wb") as ficheiro:
        shutil.copyfileobj(resposta, ficheiro)
    with zipfile.ZipFile(zip_tmp) as arquivo:
        for nome in ("ratings.csv", "movies.csv", "links.csv", "tags.csv"):
            arquivo.extract(f"{PASTA_ZIP_MOVIELENS}/{nome}", "/tmp")
            shutil.copy(f"/tmp/{PASTA_ZIP_MOVIELENS}/{nome}", f"{pasta}/{nome}")


def descarregar_imdb(pasta):
    """IMDb: descarrega os ficheiros oficiais (todos os tipos de título; a silver fica só com filmes)."""
    for nome, url in URLS_IMDB.items():
        descarregar(url, f"{pasta}/{nome}")


DOWNLOADS = {"movielens": descarregar_movielens, "imdb": descarregar_imdb}   # a TMDB não tem download


# -----------------------------------------------------------------------------
# Plano B2: mirrors públicos na Kaggle, para contas cuja rede bloqueia os sites oficiais
# -----------------------------------------------------------------------------
# Na Free Edition, o compute serverless só chega a uma lista de domínios autorizados, que varia
# entre contas. Em algumas, datasets.imdbws.com e files.grouplens.org estão bloqueados, mas a
# Kaggle (www.kaggle.com) passa. A API pública da Kaggle não pede chave para datasets públicos.
# São cópias de terceiros (verificadas em nov/2026):
#   - IMDb: dump de out/2023, em TSV sem compressão; aqui é recomprimido para .tsv.gz
#   - MovieLens 32M: edição 2023-10, os 4 CSV oficiais
#   - MovieLens small: o ml-latest-small (100 836 avaliações, 9 742 filmes)
# Nota do fallback Kaggle com o widget "small": o mirror é um ZIP de dataset, não o ZIP oficial da GroupLens.
# O código procura ratings.csv, movies.csv, links.csv e tags.csv pelos nomes exatos.
# Se faltar um ficheiro, ocorre FileNotFoundError; se o cabeçalho não corresponder, ocorre ValueError.
# O bloco captura estes erros; se continuarem a faltar ficheiros, termina com instruções para upload manual.
# Se um mirror desaparecer, troca-se o nome do dataset em MIRRORS_KAGGLE. As colunas de cada ficheiro
# são confirmadas antes de ir para o Volume; se não baterem certo, o ficheiro é rejeitado e entra o plano C.
# (fonte, versão do MovieLens) -> (dataset na Kaggle, forma de descarregar)
#   "ficheiro": um pedido por ficheiro (datasets grandes)
#   "dataset":  o dataset inteiro num só zip, e os ficheiros procurados lá dentro pelo nome (datasets pequenos)
MIRRORS_KAGGLE = {
    ("imdb", "32m"):        ("surajkarakulath/imdb-data-oct-2023", "ficheiro"),
    ("imdb", "small"):      ("surajkarakulath/imdb-data-oct-2023", "ficheiro"),
    ("movielens", "32m"):   ("liuweiq/movielens-32m-2023-10", "ficheiro"),
    ("movielens", "small"): ("whenamancodes/popular-movies-datasets-9000-movies", "dataset"),   # ml-latest-small
}
# padrão da ETAPA 2.0 -> (nome do ficheiro no dataset da Kaggle, nome com que fica no Volume)
FICHEIROS_KAGGLE = {
    "title.basics*.tsv.gz": ("title.basics.tsv", "title.basics.tsv.gz"),
    "title.ratings*.tsv.gz": ("title.ratings.tsv", "title.ratings.tsv.gz"),
    "ratings*.csv": ("ratings.csv", "ratings.csv"),
    "movies*.csv": ("movies.csv", "movies.csv"),
    "links*.csv": ("links.csv", "links.csv"),
    "tags*.csv": ("tags.csv", "tags.csv"),
}


def cabecalho_esperado(fonte, padrao):
    """Nomes das colunas que o ficheiro deve ter, tirados do schema da ETAPA 2.0."""
    schema = next(s for p, _, _, s, *_ in INGESTOES[fonte] if p == padrao)
    return re.findall(r"`([^`]+)`", schema)


def pedir_kaggle(caminho, destino):
    """Descarrega um caminho da API pública da Kaggle (sem chave) para um ficheiro local."""
    pedido = urllib.request.Request(f"https://www.kaggle.com/api/v1/datasets/download/{caminho}",
                                    headers={"User-Agent": "abd-project"})
    with urllib.request.urlopen(pedido, timeout=600) as resposta, open(destino, "wb") as ficheiro:
        shutil.copyfileobj(resposta, ficheiro)


def guardar_no_volume(origem, fonte, padrao):
    """Escreve o conteúdo em /tmp (em .gz se o nome final o pedir), confirma as colunas e só depois copia para o Volume."""
    no_volume = FICHEIROS_KAGGLE[padrao][1]
    final = f"/tmp/{no_volume}"
    with origem, (gzip.open(final, "wb") if no_volume.endswith(".gz") else open(final, "wb")) as destino:
        shutil.copyfileobj(origem, destino)
    with (gzip.open(final, "rt") if final.endswith(".gz") else open(final)) as ficheiro:
        primeira = ficheiro.readline().strip().lstrip("﻿")
    separador = "\t" if fonte == "imdb" else ","
    if primeira.split(separador) != cabecalho_esperado(fonte, padrao):
        os.remove(final)
        raise ValueError(f"{no_volume}: cabeçalho inesperado ({primeira[:80]})")
    shutil.copy(final, f"{pasta_fonte(fonte)}/{no_volume}")
    print(f"[{fonte}]   {no_volume}: {os.path.getsize(final) / 1e6:,.1f} MB copiados para o Volume")
    os.remove(final)


def descarregar_mirrors(fonte, em_falta):
    """Plano B2: descarrega da Kaggle os ficheiros em falta. Devolve False se não houver mirror para esta versão."""
    if (fonte, VERSAO_MOVIELENS) not in MIRRORS_KAGGLE:
        print(f"[{fonte}] não há mirror na Kaggle para a versão '{VERSAO_MOVIELENS}'")
        return False
    dataset, forma = MIRRORS_KAGGLE[(fonte, VERSAO_MOVIELENS)]
    print(f"[{fonte}] a tentar os mirrors públicos da Kaggle ({dataset})...")

    if forma == "dataset":
        # Dataset pequeno: um só zip; cada ficheiro é procurado pelo nome, em qualquer pasta e sem distinguir maiúsculas
        zip_tmp = f"/tmp/{dataset.replace('/', '_')}.zip"
        pedir_kaggle(dataset, zip_tmp)
        with zipfile.ZipFile(zip_tmp) as arquivo:
            membros = {os.path.basename(nome).lower(): nome for nome in arquivo.namelist()}
            for padrao in em_falta:
                no_kaggle = FICHEIROS_KAGGLE[padrao][0]
                if no_kaggle.lower() not in membros:
                    raise FileNotFoundError(f"{no_kaggle} não está no dataset {dataset}")
                guardar_no_volume(arquivo.open(membros[no_kaggle.lower()]), fonte, padrao)
        os.remove(zip_tmp)
        return True

    # Dataset grande: um pedido por ficheiro; se vier dentro de um zip, é extraído
    for padrao in em_falta:
        no_kaggle = FICHEIROS_KAGGLE[padrao][0]
        descarregado = f"/tmp/{no_kaggle}.kaggle"
        pedir_kaggle(f"{dataset}/{no_kaggle}", descarregado)
        with open(descarregado, "rb") as ficheiro:
            e_zip = ficheiro.read(2) == b"PK"
        origem = zipfile.ZipFile(descarregado).open(no_kaggle) if e_zip else open(descarregado, "rb")
        guardar_no_volume(origem, fonte, padrao)
        os.remove(descarregado)
    return True


def garantir_ficheiros(fonte):
    """Garante que os ficheiros da fonte estão no Volume antes de os carregar para a bronze."""
    pasta = pasta_fonte(fonte)
    em_falta = [padrao for padrao, *_ in INGESTOES[fonte] if not ficheiros_na_pasta(fonte, padrao)]

    # Plano A: os ficheiros já estão no Volume
    if not em_falta:
        print(f"[{fonte}] ficheiros presentes no Volume")
        return

    # Plano B: download automático (só MovieLens e IMDb)
    if fonte in DOWNLOADS:
        print(f"[{fonte}] faltam {em_falta}; a tentar o download automático...")
        try:
            DOWNLOADS[fonte](pasta)
            print(f"[{fonte}] download concluído")
            return
        except Exception as erro:
            print(f"[{fonte}] o download falhou ({type(erro).__name__}: {erro})")

    # Plano B2: os sites oficiais estão bloqueados nesta conta; tentar os mirrors públicos na Kaggle
    if fonte in DOWNLOADS:
        em_falta = [padrao for padrao, *_ in INGESTOES[fonte] if not ficheiros_na_pasta(fonte, padrao)]
        try:
            if descarregar_mirrors(fonte, em_falta):
                print(f"[{fonte}] download dos mirrors concluído (atenção: são cópias de terceiros, ver secção 2)")
                return
        except Exception as erro:
            print(f"[{fonte}] os mirrors também falharam ({type(erro).__name__}: {erro})")

    # Os planos B e B2 podem ter trazido parte dos ficheiros: o que falta é contado outra vez
    em_falta = [padrao for padrao, *_ in INGESTOES[fonte] if not ficheiros_na_pasta(fonte, padrao)]
    if not em_falta:
        return

    # Se a bronze já tem estes dados de uma execução anterior, continua-se com esse histórico
    tabelas = [tabela for padrao, tabela, *_ in INGESTOES[fonte] if padrao in em_falta]
    if all(spark.catalog.tableExists(nome_tabela(BRONZE, t)) for t in tabelas):
        print(f"[{fonte}] sem ficheiros novos; mantém-se o histórico já guardado na bronze")
        return

    # Plano C: upload manual
    motivo = ("O download automático não foi possível: nesta conta, a rede da Free Edition não chega aos sites "
              "oficiais nem aos mirrors (só deixa passar uma lista de domínios autorizados).")
    # [TMDB desativada] if fonte == "tmdb": motivo = "Os dados da TMDB vêm de uma API com chave pessoal e não são descarregados pelo notebook."
    raise FileNotFoundError(f"{motivo}\nCarrega os ficheiros {em_falta} em Catalog > {CATALOGO} > {BRONZE} > "
                            f"{VOLUME} > {PASTAS_FONTES[fonte]} > Upload to this volume e volta a correr o notebook.")


# -----------------------------------------------------------------------------
# Passo 2: carregar para a bronze com Auto Loader
# -----------------------------------------------------------------------------
def com_auditoria(df, fonte):
    """Colunas de auditoria comuns às duas formas de ingestão: fonte, ficheiro, data do ficheiro, lote e momento."""
    return (df.withColumn("_source", F.lit(fonte))
              .withColumn("_source_file", F.col("_metadata.file_path"))
              .withColumn("_file_modified", F.col("_metadata.file_modification_time"))
              .withColumn("_batch_id", F.lit(LOTE))
              .withColumn("_ingested_at", F.current_timestamp()))


def bronze_auto_loader(fonte, padrao, tabela, formato, schema, opcoes, descricao):
    """Lê para a bronze só os ficheiros novos da pasta da fonte e acrescenta-os à tabela (append)."""
    destino = nome_tabela(BRONZE, tabela)
    consulta = (
        spark.readStream.format("cloudFiles")                   # cloudFiles = Auto Loader
        .option("cloudFiles.format", formato)                   # csv ou json
        .option("pathGlobFilter", padrao)                       # só os ficheiros desta tabela
        .options(**opcoes)                                      # separador, aspas, cabeçalho...
        .schema(schema)                                         # tudo em texto: sem inferência de tipos
        .load(pasta_fonte(fonte))
        .transform(lambda df: com_auditoria(df, fonte))         # de onde veio cada linha e quando entrou
        .writeStream
        .option("checkpointLocation", f"{PASTA_CHECKPOINTS}/{tabela}")   # lista dos ficheiros já lidos
        .option("mergeSchema", "true")
        .trigger(availableNow=True)                             # processa o que há de novo e pára
        .toTable(destino)
    )
    consulta.awaitTermination()                                 # espera que o stream termine
    # Linhas novas = linhas da tabela marcadas com o lote desta execução
    # (o progresso do stream nem sempre traz este número no compute serverless)
    novas = spark.table(destino).where(F.col("_batch_id") == LOTE).count()
    definir_descricao(destino, descricao)
    print(f"[bronze] {destino}: +{novas:,} linhas novas (lote {LOTE})")

# [TMDB desativada] leitura batch incremental, usada só na TMDB
# # -----------------------------------------------------------------------------
# # Alternativa sem Auto Loader: leitura batch incremental (usada na TMDB, ver secção 2)
# # -----------------------------------------------------------------------------
# def bronze_batch(fonte, padrao, tabela, formato, schema, opcoes, descricao):
    # """Lê para a bronze, em batch, só os ficheiros que ainda não estão na tabela (append).

    # Faz de forma explícita o que o checkpoint do Auto Loader faz sozinho: compara cada ficheiro
    # (caminho e data de modificação) com os que já estão na bronze e fica só com os novos."""
    # destino = nome_tabela(BRONZE, tabela)
    # lidos_agora = com_auditoria(
        # spark.read.format(formato)
        # .options(**opcoes)
        # .option("pathGlobFilter", padrao)                       # só os ficheiros desta tabela
        # .schema(schema)
        # .load(f"{RAIZ_RAW}/{fonte}"),
        # fonte)

    # # left_anti: fica só com as linhas cujos ficheiros ainda não estão na tabela
    # if spark.catalog.tableExists(destino):
        # ja_lidos = spark.table(destino).select("_source_file", "_file_modified").distinct()
        # lidos_agora = lidos_agora.join(F.broadcast(ja_lidos), ["_source_file", "_file_modified"], "left_anti")

    # lidos_agora.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(destino)
    # novas = spark.table(destino).where(F.col("_batch_id") == LOTE).count()
    # definir_descricao(destino, descricao)
    # print(f"[bronze] {destino}: +{novas:,} linhas novas (lote {LOTE}, leitura batch)")


def verificar_versao_movielens():
    """Impede que a bronze misture as duas versões do MovieLens (ids de utilizadores e avaliações diferentes)."""
    destino = nome_tabela(BRONZE, "ml_ratings")
    if not spark.catalog.tableExists(destino):
        return
    pasta = PASTAS_FONTES["movielens"]
    outra_versao = spark.table(destino).where(~F.col("_source_file").contains(f"/{pasta}/")).limit(1).count()
    if outra_versao:
        raise RuntimeError(f"A bronze do MovieLens tem dados de outra versão (o widget pede '{VERSAO_MOVIELENS}'). "
                           "Corre o notebook 00_reset e volta a correr este notebook. "
                           "Os ficheiros das duas versões ficam no Volume, em pastas separadas.")


def bronze_fonte(fonte):
    """Pipeline da bronze para uma fonte: garantir os ficheiros e carregar cada tabela com Auto Loader."""
    if fonte == "movielens":
        verificar_versao_movielens()
    garantir_ficheiros(fonte)
    for padrao, tabela, formato, schema, opcoes, descricao in INGESTOES[fonte]:
        bronze_auto_loader(fonte, padrao, tabela, formato, schema, opcoes, descricao)
        # [TMDB desativada] com modo="batch", a TMDB usava bronze_batch em vez de bronze_auto_loader


In [0]:
# =============================================================================
# ETAPA 2.1 — BRONZE · TMDB (3 tabelas, JSON), em leitura batch incremental  [TMDB desativada]
# Os ficheiros vêm de uma extração única da API, carregada à mão: não há um fluxo de
# ficheiros novos que justifique um stream (ver secção 2).
# Não corre: a TMDB não tem download automático nem Auto Loader (ver secção 2).
# Para voltar a ativar: retirar o '# ' do início de cada linha.

# # =============================================================================
# bronze_fonte("tmdb", modo="batch")

# # Alternativa com Auto Loader, igual às outras fontes (o Auto Loader também lê JSON):
# # bronze_fonte("tmdb")
# # Se mudarem de método com dados já carregados, esvaziem primeiro as 3 tabelas bronze da TMDB
# # (secção 2.5, reparar_bronze sem ficheiro): cada método tem a sua própria memória dos ficheiros lidos.

In [0]:
# =============================================================================
# ETAPA 2.2 — BRONZE · IMDb (2 tabelas, TSV comprimido)
# =============================================================================
bronze_fonte("imdb")

In [0]:
# =============================================================================
# ETAPA 2.3 — BRONZE · MovieLens (4 tabelas, CSV)
# =============================================================================
bronze_fonte("movielens")

### 2.4 Perfil dos dados brutos

O gráfico mostra quantas linhas entraram em cada tabela bronze em cada lote. Com duas leituras da fonte aparecem duas cores na mesma barra: é a evidência de que a bronze guarda o histórico de leituras. Visível apenas se se correr o notebook uma segunda vez.

In [0]:
# =============================================================================
# ETAPA 2.4 — Figura 1: linhas por tabela bronze e por lote de ingestão
# =============================================================================
# 1) Contar as linhas de cada tabela bronze por lote e juntar tudo numa só tabela
partes = [spark.table(nome_tabela(BRONZE, t)).groupBy("_batch_id").count().withColumn("tabela", F.lit(t))
          for tabelas in TABELAS_BRONZE.values() for t in tabelas]
contagens = partes[0]
for parte in partes[1:]:
    contagens = contagens.unionByName(parte)
lotes = contagens.toPandas().pivot_table(index="tabela", columns="_batch_id", values="count", fill_value=0)


# 2) Eixo em milhões: 1 000 000 -> 1M, 100 000 -> 0,1M (abaixo de 1000 mostra o número normal)
def em_milhoes(x, _):
    return f"{x:.0f}" if x < 1000 else f"{x / 1e6:g}M".replace(".", ",")


# 3) Desenhar uma barra por lote
fig, ax = plt.subplots(figsize=(10, 4.5))
lotes.plot.barh(ax=ax, colormap="tab10", width=0.8)
ax.set_xscale("log")
ax.set_xlim(left=0.5)
ax.xaxis.set_major_formatter(FuncFormatter(em_milhoes))
ax.set_xlabel("Linhas, em milhões (M), escala logarítmica")
ax.set_ylabel("")
ax.set_title("Figura 1. Linhas por tabela bronze e por lote de ingestão")
ax.legend(title="Lote", fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()
print(f"Lotes distintos na bronze: {lotes.shape[1]}")

### 2.5 Reparar a bronze (só quando é preciso)

A bronze só recebe linhas (append) e nunca é reescrita. Às vezes é preciso corrigi-la: um ficheiro chegou com dados errados, ou uma opção de leitura estava mal (ex.: o separador) e uma tabela inteira ficou mal lida.

**Porque é que o `RESTORE` do Delta Lake não chega sozinho.** O `RESTORE TABLE ... TO VERSION AS OF n` volta a pôr a tabela como estava numa versão anterior, mas **não mexe no checkpoint do Auto Loader**. Os ficheiros lidos depois dessa versão desaparecem da tabela e continuam registados no checkpoint como "já lidos": nunca mais voltariam a entrar. Por isso, a tabela e a memória dos ficheiros lidos têm de ser reparadas **em conjunto**.

| Situação | O que fazer |
|---|---|
| **Um ficheiro trazia dados errados** | 1) Carregar o ficheiro corrigido com um **nome novo** (ex.: `ratings_2026-11-14.csv`). 
| | 2) Correr a etapa 2: entra só esse ficheiro. 
| | 3) Apagar da bronze as linhas do ficheiro errado: `reparar_bronze("ml_ratings", ficheiro="ratings.csv", confirmar=True)` |
| **Uma tabela inteira ficou mal lida** | 1) Corrigir a opção de leitura em `INGESTOES` (célula 2.0). 
| | 2) `reparar_bronze("ml_ratings", confirmar=True)`: esvazia a tabela e apaga o seu checkpoint. 
| | 3) Correr a etapa 2: todos os ficheiros da pasta voltam a ser lidos |
| **Uma reparação correu mal** | `RESTORE TABLE` para a versão indicada pela própria `reparar_bronze` e, se a tabela usa Auto Loader, repetir o caso anterior para realinhar o checkpoint |

Em todos os casos, a silver e a gold ficam corretas na execução seguinte, porque são sempre reconstruídas a partir da bronze.

A função seguinte só é **definida**: não é executada no *Run all*. Por omissão (`confirmar=False`) só mostra o que faria. Usa `DELETE` em vez de `DROP TABLE`, para que a reparação fique no histórico Delta e possa ser desfeita.

In [0]:
# =============================================================================
# ETAPA 2.5 — Reparação da bronze: tabela e memória dos ficheiros lidos, sempre em conjunto
# (só é definida aqui; usa-se à mão, numa célula nova, quando for preciso)
# =============================================================================
def reparar_bronze(tabela, ficheiro=None, confirmar=False):
    """Corrige uma tabela bronze sem a desalinhar da memória dos ficheiros lidos.

    ficheiro=None   -> esvazia a tabela e apaga o seu checkpoint do Auto Loader; na execução
                       seguinte da etapa 2, todos os ficheiros da pasta voltam a ser lidos.
    ficheiro="nome" -> apaga só as linhas que vieram desse ficheiro (o ficheiro corrigido deve
                       ser carregado com outro nome, para entrar como novo).
    confirmar=False -> simulação: mostra o que seria feito e não altera nada."""
    destino = nome_tabela(BRONZE, tabela)
    if ficheiro:
        condicao = f"_source_file LIKE '%/{ficheiro}'"
        acao = f"apagar as linhas vindas de '{ficheiro}'"
    else:
        condicao = "true"
        acao = "esvaziar a tabela e apagar o seu checkpoint"
    afetadas = spark.table(destino).where(condicao).count()
    print(f"[reparação] {destino}: {acao} ({afetadas:,} linhas)")
    if not confirmar:
        print("[reparação] simulação: nada foi alterado. Para aplicar, repetir com confirmar=True.")
        return

    # Versão atual, para poder desfazer com RESTORE
    versao = spark.sql(f"DESCRIBE HISTORY {destino} LIMIT 1").first()["version"]
    spark.sql(f"DELETE FROM {destino} WHERE {condicao}")
    if not ficheiro:
        # Sem checkpoint, o Auto Loader volta a ler todos os ficheiros.
        try:
            dbutils.fs.rm(f"{PASTA_CHECKPOINTS}/{tabela}", True)
        except Exception:
            pass   # a pasta do checkpoint não existe (tabela lida em batch ou nunca carregada)
    print(f"[reparação] concluída. Para desfazer: RESTORE TABLE {destino} TO VERSION AS OF {versao}")
    print("[reparação] Correr de novo a etapa 2 e as seguintes para atualizar a bronze, a silver e a gold.")


# Exemplos de utilização (comentados: não correm no Run all)
# reparar_bronze("ml_ratings", ficheiro="ratings.csv")                   # simulação
# reparar_bronze("ml_ratings", ficheiro="ratings.csv", confirmar=True)   # apaga as linhas desse ficheiro
# reparar_bronze("ml_tags", confirmar=True)                              # tabela inteira

## 3. Data Preparation: Silver (CRISP-DM fase 3)

A silver é reconstruída a partir de **todo** o histórico da bronze e fica com a versão mais recente de cada chave (função `mais_recente`). Por isso está sempre correta, mesmo depois de várias leituras da fonte.

Regras aplicadas: `\N` e textos vazios passam a nulo; tipos convertidos com `try_cast`; deduplicação determinística; géneros em array; datas validadas.

In [0]:
# =============================================================================
# ETAPA 3.0 — Funções da silver (limpeza, tipos, deduplicação e escrita)
# =============================================================================
def mais_recente(df, chaves, ordem=("_ingested_at", "_source_file")):
    """Fica só com a versão mais recente de cada chave.

    Numera as linhas de cada chave por ordem decrescente das colunas em 'ordem' e guarda a primeira.
    O resultado é determinístico porque, depois do momento da ingestão, o ficheiro de origem
    (e, quando é preciso, outras colunas) desempatam linhas lidas no mesmo lote."""
    janela = Window.partitionBy(*chaves).orderBy(*[F.col(c).desc() for c in ordem])
    return df.withColumn("_n", F.row_number().over(janela)).where("_n = 1").drop("_n")


def marcar_nulos(df, marcadores=("\\N",)):
    """Converte em nulo os marcadores de 'valor em falta' (ex.: \\N do IMDb) em todas as colunas de texto."""
    colunas_texto = {c for c, tipo in df.dtypes if tipo == "string"}
    return df.select(*[F.when(F.col(c).isin(*marcadores), F.lit(None)).otherwise(F.col(c)).alias(c)
                       if c in colunas_texto else F.col(c) for c in df.columns])


def converter(coluna, tipo):
    """Converte o tipo de uma coluna; valores inválidos ficam nulos em vez de pararem a execução."""
    return F.expr(f"try_cast(`{coluna}` AS {tipo})")


def data_segura(coluna):
    """Texto 'aaaa-mm-dd' para data; qualquer outro valor fica nulo."""
    return F.when(F.col(coluna).rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(F.col(coluna)))


def chave_imdb(coluna):
    """Constrói o tconst do IMDb (tt0114709) a partir do imdbId do MovieLens.
    Acrescenta zeros à esquerda até 7 dígitos, mas nunca corta: os filmes recentes têm 8 (tt10872600)."""
    c = F.trim(F.col(coluna))
    return (F.when(c.isNull() | (c == ""), F.lit(None))
             .otherwise(F.concat(F.lit("tt"), F.when(F.length(c) < 7, F.lpad(c, 7, "0")).otherwise(c))))


def verificar_chave_unica(df, chave, rotulo):
    """Controlo de qualidade: pára a execução se a chave tiver valores repetidos."""
    chaves = [chave] if isinstance(chave, str) else list(chave)
    total, distintas = df.agg(F.count("*"), F.countDistinct(*chaves)).first()
    if total != distintas:
        raise ValueError(f"{rotulo}: chave {chaves} repetida ({total:,} linhas, {distintas:,} distintas)")
    print(f"[qualidade] {rotulo}: chave {chaves} única ({total:,} linhas)")


def gravar_delta(df, camada, tabela, cluster_by=None, descricao=None):
    """Reescreve uma tabela silver ou gold a partir da camada anterior e devolve-a.

    Com cluster_by, a tabela é criada com Liquid Clustering. Como essa opção se declara em SQL,
    o DataFrame é primeiro registado como vista temporária. A versão anterior da tabela fica
    no histórico Delta (time travel)."""
    destino = nome_tabela(camada, tabela)
    if cluster_by:
        vista = f"_tmp_{uuid.uuid4().hex[:8]}"
        df.createOrReplaceTempView(vista)
        spark.sql(f"CREATE OR REPLACE TABLE {destino} CLUSTER BY ({', '.join(cluster_by)}) AS SELECT * FROM {vista}")
        spark.catalog.dropTempView(vista)
    else:
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
    if descricao:
        definir_descricao(destino, descricao)
    print(f"[{camada}] {destino}: gravada")
    return spark.table(destino)

In [0]:
# =============================================================================
# ETAPA 3.1 — SILVER · TMDB: extrair o JSON e ficar com a versão mais recente de cada filme  [TMDB desativada]
# =============================================================================
# Schemas do JSON da TMDB (só os campos que usamos; os restantes são ignorados)
# Não corre: depende das tabelas bronze da TMDB, que não são carregadas nesta versão.
# Para voltar a ativar: retirar o '# ' do início de cada linha.

# SCHEMA_FILME_LISTA = ("id long, title string, original_title string, original_language string, overview string, "
                      # "release_date string, popularity double, vote_average double, vote_count long")
# SCHEMA_DETALHE = ("id long, imdb_id string, title string, original_title string, original_language string, "
                  # "overview string, release_date string, runtime int, budget long, revenue long, status string, "
                  # "popularity double, vote_average double, vote_count long, "
                  # "genres array<struct<id:int,name:string>>")

# # 1) Filmes das listas: uma linha por filme e por lista, com o JSON transformado em colunas
# listas = (spark.table(nome_tabela(BRONZE, "tmdb_movie_lists"))
          # .select("list_name", "page", "_ingested_at", "_source_file",
                  # F.from_json("payload", SCHEMA_FILME_LISTA).alias("m"))
          # .select("list_name", "page", "_ingested_at", "_source_file", F.col("m.id").alias("tmdb_id"), "m.*")
          # .where(F.col("tmdb_id").isNotNull()))

# # 2) Em que listas aparece cada filme (popular, top_rated...)
# listas_por_filme = listas.groupBy("tmdb_id").agg(F.sort_array(F.collect_set("list_name")).alias("lists"))

# # 3) Uma linha por filme: a versão mais recente. Um filme que aparece em duas listas no mesmo lote
# #    tem o mesmo momento de ingestão, por isso a lista e a página servem de desempate.
# tmdb_movies = (mais_recente(listas, ["tmdb_id"], ("_ingested_at", "_source_file", "list_name", "page"))
               # .join(listas_por_filme, "tmdb_id")
               # .select("tmdb_id", "title", "original_title", "original_language", "overview",
                       # data_segura("release_date").alias("release_date"),
                       # "popularity", "vote_average", "vote_count", "lists"))
# verificar_chave_unica(tmdb_movies, "tmdb_id", "silver.tmdb_movies")
# gravar_delta(tmdb_movies, SILVER, "tmdb_movies",
             # descricao="TMDB: filmes das listas, um por tmdb_id, com as listas em que aparecem")

# # 4) Detalhe de cada filme (orçamento, receita, duração, géneros): a versão mais recente
# detalhes = (spark.table(nome_tabela(BRONZE, "tmdb_movie_details"))
            # .select("_ingested_at", "_source_file", F.from_json("payload", SCHEMA_DETALHE).alias("d"))
            # .select("_ingested_at", "_source_file", "d.*")
            # .where(F.col("id").isNotNull()))
# tmdb_detalhes = (mais_recente(marcar_nulos(detalhes, marcadores=("",)), ["id"])
                 # .select(F.col("id").alias("tmdb_id"), "imdb_id", "title", "original_title",
                         # "original_language", "overview", data_segura("release_date").alias("release_date"),
                         # F.col("runtime").alias("runtime_minutes"), "budget", "revenue", "status",
                         # "popularity", "vote_average", "vote_count", "genres"))
# verificar_chave_unica(tmdb_detalhes, "tmdb_id", "silver.tmdb_movie_details")
# tmdb_detalhes = gravar_delta(tmdb_detalhes, SILVER, "tmdb_movie_details",
                             # descricao="TMDB: detalhe tipado (orçamento, receita, duração, géneros), versão mais recente")

# # 5) Relação filme-género: os géneros vêm como lista de pares (id, nome); explode passa a uma linha por género
# tmdb_generos = (tmdb_detalhes.select("tmdb_id", F.explode("genres").alias("g"))
                # .select("tmdb_id", F.col("g.id").alias("genre_id"), F.col("g.name").alias("genre_name")))
# gravar_delta(tmdb_generos, SILVER, "tmdb_movie_genres", descricao="TMDB: relação filme-género")

In [0]:
# =============================================================================
# ETAPA 3.2 — SILVER · IMDb: só filmes não adultos, com tipos corretos, nota e votos
# =============================================================================
# 1) Versão mais recente de cada título, com o \N do IMDb convertido em nulo
basics = mais_recente(marcar_nulos(spark.table(nome_tabela(BRONZE, "imdb_title_basics"))), ["tconst"])
ratings = mais_recente(marcar_nulos(spark.table(nome_tabela(BRONZE, "imdb_title_ratings"))), ["tconst"])

# 2) Nota e nº de votos, convertidos para número
imdb_notas = ratings.select("tconst",
                            converter("averageRating", "double").alias("imdb_rating"),
                            converter("numVotes", "bigint").alias("imdb_votes"))

# 3) Filmes (titleType = movie) não adultos, com a nota (left join: filmes sem votos ficam com nota nula)
imdb_movies = (
    basics.where(F.col("titleType") == "movie")
    .select("tconst",
            F.col("primaryTitle").alias("title"),
            F.col("originalTitle").alias("original_title"),
            F.coalesce(F.col("isAdult") == "1", F.lit(False)).alias("is_adult"),
            converter("startYear", "int").alias("start_year"),
            converter("runtimeMinutes", "int").alias("runtime_minutes"),
            F.split("genres", ",").alias("genres"))
    .where(~F.col("is_adult"))
    .join(imdb_notas, "tconst", "left")
)
verificar_chave_unica(imdb_movies, "tconst", "silver.imdb_movies")
imdb_movies = gravar_delta(imdb_movies, SILVER, "imdb_movies", cluster_by=["start_year"],
                           descricao="IMDb: filmes não adultos, tipados, com nota e votos (Liquid Clustering por start_year)")

# 4) Relação filme-género (uma linha por filme e género)
gravar_delta(imdb_movies.select("tconst", F.explode("genres").alias("genre")), SILVER, "imdb_movie_genres",
             descricao="IMDb: relação filme-género (nomes em inglês)")

In [0]:
# =============================================================================
# ETAPA 3.3 — SILVER · MovieLens: avaliações, filmes, ligações às outras fontes e tags
# =============================================================================
PADRAO_ANO = r"\((\d{4})\)\s*$"   # o ano vem no fim do título: "Toy Story (1995)"

# 1) Avaliações: uma por (utilizador, filme), entre 0,5 e 5; se houver repetidas fica a mais recente
ml_avaliacoes = (
    spark.table(nome_tabela(BRONZE, "ml_ratings"))
    .select(converter("userId", "int").alias("user_id"),
            converter("movieId", "int").alias("movie_id"),
            converter("rating", "double").alias("rating"),
            F.timestamp_seconds(converter("timestamp", "bigint")).alias("rated_at"),
            "_ingested_at", "_source_file")
    .where(F.col("user_id").isNotNull() & F.col("movie_id").isNotNull() & F.col("rating").between(0.5, 5.0))
)
ml_avaliacoes = (mais_recente(ml_avaliacoes, ["user_id", "movie_id"], ("_ingested_at", "rated_at", "_source_file"))
                 .drop("_ingested_at", "_source_file"))
verificar_chave_unica(ml_avaliacoes, ["user_id", "movie_id"], "silver.ml_ratings")
gravar_delta(ml_avaliacoes, SILVER, "ml_ratings", cluster_by=["movie_id"],
             descricao="MovieLens: uma avaliação por utilizador e filme (escala 0.5 a 5)")

# 2) Filmes: título sem o ano, ano extraído do título e géneros em array (sem "(no genres listed)")
ml_filmes = (
    mais_recente(spark.table(nome_tabela(BRONZE, "ml_movies")), ["movieId"])
    .select(converter("movieId", "int").alias("movie_id"),
            F.trim(F.regexp_replace("title", PADRAO_ANO, "")).alias("title"),
            F.when(F.regexp_extract("title", PADRAO_ANO, 1) != "",
                   F.regexp_extract("title", PADRAO_ANO, 1).cast("int")).alias("year"),
            F.filter(F.split("genres", r"\|"), lambda g: (g != "") & (g != "(no genres listed)")).alias("genres"))
    .where(F.col("movie_id").isNotNull())
)
verificar_chave_unica(ml_filmes, "movie_id", "silver.ml_movies")
gravar_delta(ml_filmes, SILVER, "ml_movies", descricao="MovieLens: filmes com título, ano e géneros em array")

# 3) Ligações: a ponte entre as três fontes (movie_id do MovieLens, tconst do IMDb, tmdb_id da TMDB)
ml_ligacoes = (
    mais_recente(spark.table(nome_tabela(BRONZE, "ml_links")), ["movieId"])
    .select(converter("movieId", "int").alias("movie_id"),
            chave_imdb("imdbId").alias("tconst"))
            # [TMDB desativada] chave da TMDB, só usada para juntar a TMDB
            # converter("tmdbId", "int").alias("tmdb_id"))
    .where(F.col("movie_id").isNotNull())
)
verificar_chave_unica(ml_ligacoes, "movie_id", "silver.ml_links")
gravar_delta(ml_ligacoes, SILVER, "ml_links", descricao="MovieLens: ponte movie_id <-> tconst (IMDb)")

# 4) Tags de texto livre: em minúsculas e sem repetições por utilizador e filme
ml_tags = (
    spark.table(nome_tabela(BRONZE, "ml_tags"))
    .select(converter("userId", "int").alias("user_id"),
            converter("movieId", "int").alias("movie_id"),
            F.lower(F.trim("tag")).alias("tag"),
            F.timestamp_seconds(converter("timestamp", "bigint")).alias("tagged_at"))
    .where(F.col("tag").isNotNull() & (F.col("tag") != ""))
    .dropDuplicates(["user_id", "movie_id", "tag"])
)
gravar_delta(ml_tags, SILVER, "ml_tags", descricao="MovieLens: tags de texto livre normalizadas (minúsculas)")

### 3.4 Silver integrada: `silver.movies_unified`

Junta o IMDb e o MovieLens por chave e preserva no código uma extensão para os metadados complementares. 

O IMDb é a tabela grande e as restantes são pequenas, por isso usam-se **Broadcast Hash Joins**: a tabela pequena é enviada a todas as máquinas e evita-se o *shuffle* (o `explain` abaixo mostra-o).

Quando os dados complementares estão carregados, os valores `0` de orçamento, receita e votos significam "desconhecido" e passam a nulo. A nota do MovieLens (0,5 a 5) é multiplicada por 2 para ficar na escala do IMDb (1 a 10).

In [0]:
# =============================================================================
# ETAPA 3.4 — SILVER INTEGRADA: IMDb + MovieLens numa só tabela ([TMDB desativada]: a junção com a TMDB está comentada)
# =============================================================================
imdb = spark.table(nome_tabela(SILVER, "imdb_movies"))
ligacoes = spark.table(nome_tabela(SILVER, "ml_links")).where(F.col("tconst").isNotNull())
# [TMDB desativada] tmdb = spark.table(nome_tabela(SILVER, "tmdb_movie_details"))

# 1) Média e nº de avaliações MovieLens por filme
ml_resumo = (spark.table(nome_tabela(SILVER, "ml_ratings"))
             .groupBy("movie_id")
             .agg(F.avg("rating").alias("ml_avg_rating"), F.count("*").alias("ml_num_ratings")))

# 2) [TMDB desativada] Colunas da TMDB, com os zeros ("desconhecido") convertidos em nulo
# tmdb_sel = tmdb.select(
#     "tmdb_id", "original_language", "overview",
#     F.when(F.col("vote_count") > 0, F.col("vote_average")).alias("tmdb_vote_average"),
#     F.when(F.col("vote_count") > 0, F.col("vote_count")).alias("tmdb_vote_count"),
#     F.col("popularity").alias("tmdb_popularity"),
#     F.when(F.col("budget") > 0, F.col("budget")).alias("budget"),
#     F.when(F.col("revenue") > 0, F.col("revenue")).alias("revenue"),
#     F.lit(True).alias("in_tmdb"),
# )

# 3) Junções: IMDb (grande) com as tabelas pequenas, enviadas por broadcast
unificada = (
    imdb.join(F.broadcast(ligacoes), "tconst", "inner")      # só filmes que existem no IMDb e no MovieLens
        # [TMDB desativada] .join(F.broadcast(tmdb_sel), "tmdb_id", "left")   # + TMDB, quando existir
        .join(F.broadcast(ml_resumo), "movie_id", "left")    # + avaliações MovieLens, quando existirem
        .select("movie_id", "tconst", "title", "start_year",
                (F.floor(F.col("start_year") / 10) * 10).cast("int").alias("decade"),
                "runtime_minutes", "genres", "imdb_rating", "imdb_votes",
                F.round("ml_avg_rating", 3).alias("ml_avg_rating"),
                F.round(F.col("ml_avg_rating") * 2, 3).alias("ml_rating_10"), "ml_num_ratings")
        # [TMDB desativada] colunas da TMDB: "tmdb_id", "tmdb_vote_average", "tmdb_vote_count", "tmdb_popularity",
        #     "budget", "revenue", "original_language", "overview",
        #     F.coalesce(F.col("in_tmdb"), F.lit(False)).alias("in_tmdb")
)

# 4) Plano de execução: procurar BroadcastHashJoin (evidência para o relatório)
unificada.explain(mode="formatted")

verificar_chave_unica(unificada, "movie_id", "silver.movies_unified")
gravar_delta(unificada, SILVER, "movies_unified", cluster_by=["decade", "start_year"],
             descricao="Filmes integrados (IMDb + MovieLens); ml_rating_10 = nota MovieLens na escala 1-10")


### 3.5 Qualidade dos dados

Cada regra conta as linhas que a violam. O resultado é acrescentado a `gold.dq_results` (em `append`, com o lote), para se ver a evolução da qualidade ao longo das execuções.

In [0]:
# =============================================================================
# ETAPA 3.5 — QUALIDADE: 5 regras sobre a silver, guardadas em gold.dq_results ([TMDB desativada]: 4 regras comentadas)
# =============================================================================
# (tabela, regra, condição SQL que identifica as linhas que violam a regra)
REGRAS_QUALIDADE = [
    ("silver.imdb_movies",        "nota IMDb fora de 1-10",          "imdb_rating IS NOT NULL AND imdb_rating NOT BETWEEN 1 AND 10"),
    ("silver.imdb_movies",        "ano de estreia em falta",         "start_year IS NULL"),
    ("silver.ml_ratings",         "nota MovieLens fora de 0.5-5",    "rating NOT BETWEEN 0.5 AND 5"),
    ("silver.ml_links",           "filme sem tconst",                "tconst IS NULL"),
    # [TMDB desativada] 
    # ("silver.ml_links",           "filme sem tmdb_id",               "tmdb_id IS NULL"),
    # ("silver.tmdb_movie_details", "orçamento ou receita negativos",  "budget < 0 OR revenue < 0"),
    # ("silver.tmdb_movie_details", "orçamento suspeito (< 1000 USD)", "budget > 0 AND budget < 1000"),
    ("silver.movies_unified",     "filme sem avaliações MovieLens",  "ml_num_ratings IS NULL"),
    # [TMDB desativada] 
    # ("silver.movies_unified",     "filme sem detalhe TMDB",          "NOT in_tmdb"),
]

# 1) Para cada regra: nº total de linhas e nº de linhas que a violam
resultados = []
for tabela, regra, condicao in REGRAS_QUALIDADE:
    total, violacoes = (spark.table(f"{CATALOGO}.{tabela}")
                        .agg(F.count("*"), F.sum(F.expr(condicao).cast("int"))).first())
    resultados.append((tabela, regra, int(total), int(violacoes or 0)))

# 2) Percentagem de violações. try_divide devolve nulo se a tabela estiver vazia, em vez de dar
#    erro de divisão por zero (o compute serverless usa o modo ANSI, que pára nesse caso).
qualidade = (spark.createDataFrame(resultados, "tabela string, regra string, linhas long, violacoes long")
             .withColumn("pct_violacoes", F.round(F.expr("try_divide(100 * violacoes, linhas)"), 2))
             .withColumn("_batch_id", F.lit(LOTE))
             .withColumn("_checked_at", F.current_timestamp()))

# 3) Acrescentar ao histórico da qualidade
qualidade.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(nome_tabela(GOLD, "dq_results"))
display(qualidade)

## 4. Modeling: tabelas Gold (CRISP-DM fase 4)

Populações de análise, ambas com limiares mínimos de votos para evitar ruído (widgets `min_*`):

* **`base`**: filmes com nota IMDb e MovieLens fiáveis. Usada para comparar o IMDb com o MovieLens.
* **`base_cmp`**: subconjunto da `base` com notas fiáveis das fontes incluídas na comparação; permite comparar as fontes nos mesmos filmes quando a extensão correspondente está ativa.

Diferença (*gap*) = nota IMDb − nota MovieLens (escala 1-10). Um gap positivo significa que o IMDb é mais generoso.

As secções 4.1 a 4.3 criam **agregados** prontos para cada pergunta. A secção 4.4 guarda os mesmos dados **ao detalhe**, num star schema.

In [0]:
# =============================================================================
# ETAPA 4.0 — Populações de análise e funções de estatística
# =============================================================================
def populacoes_analise():
    """Devolve (unificada, base): a silver integrada e a população de análise.
    [TMDB desativada] A segunda população, base_cmp (filmes também com nota TMDB), está comentada."""
    unificada = spark.table(nome_tabela(SILVER, "movies_unified"))
    base = (unificada
            .where((F.col("imdb_votes") >= MIN_VOTOS_IMDB)
                   & (F.col("ml_num_ratings") >= MIN_AVALIACOES_ML)
                   & F.col("decade").isNotNull())
            .withColumn("gap_imdb_ml", F.col("imdb_rating") - F.col("ml_rating_10")))
    # [TMDB desativada]
    # base_cmp = (base.where(F.col("tmdb_vote_count") >= MIN_VOTOS_TMDB)
    #             .withColumn("gap_imdb_tmdb", F.col("imdb_rating") - F.col("tmdb_vote_average")))
    return unificada, base


def estatisticas(coluna, prefixo):
    """Média, desvio-padrão e nº de valores não nulos de uma coluna.
    O nº de valores (n_<prefixo>) é o que se usa no intervalo de confiança da média."""
    return [F.round(F.avg(coluna), 3).alias(f"avg_{prefixo}"),
            F.round(F.stddev(coluna), 3).alias(f"sd_{prefixo}"),
            F.count(coluna).alias(f"n_{prefixo}")]


unificada, base = populacoes_analise()


In [0]:
# =============================================================================
# ETAPA 4.1 — GOLD · uma tabela por hipótese, mais tags ([TMDB desativada]: H3, H4 e retorno financeiro comentados)
# =============================================================================
# H1 e H5: nota média por fonte e por década, nos mesmos filmes (base)
por_decada = (base.groupBy("decade")
              .agg(F.count("*").alias("n_movies"),
                   *estatisticas("imdb_rating", "imdb"), *estatisticas("ml_rating_10", "ml"))
                   # [TMDB desativada] , *estatisticas("tmdb_vote_average", "tmdb")
              .where("n_movies >= 5"))
gravar_delta(por_decada, GOLD, "ratings_by_decade",
             descricao="Nota média e desvio-padrão do IMDb e do MovieLens por década, nos mesmos filmes (base)")

# H2: gap IMDb - MovieLens por género (um filme com 3 géneros conta nos 3)
por_genero = (base.withColumn("genre", F.explode("genres"))
              .groupBy("genre")
              .agg(F.count("*").alias("n_movies"), *estatisticas("gap_imdb_ml", "gap"),
                   F.round(F.avg("imdb_rating"), 3).alias("avg_imdb"),
                   F.round(F.avg("ml_rating_10"), 3).alias("avg_ml"))
              .where("n_movies >= 20"))
gravar_delta(por_genero, GOLD, "gap_by_genre", descricao="Diferença IMDb - MovieLens por género (base, mín. 20 filmes)")

# [TMDB desativada] H3 (idioma) e H4 (orçamento) precisam da TMDB
# # H3: gap por idioma original (informação da TMDB)
# por_idioma = (base_cmp.groupBy("original_language")
              # .agg(F.count("*").alias("n_movies"), *estatisticas("gap_imdb_ml", "gap_ml"),
                   # *estatisticas("gap_imdb_tmdb", "gap_tmdb"))
              # .where("n_movies >= 10"))
# gravar_delta(por_idioma, GOLD, "gap_by_language", descricao="Diferenças de nota por idioma original (base_cmp, mín. 10 filmes)")

# # H4: gap e consenso (diferença absoluta) por escalão de orçamento
# por_orcamento = (base.where(F.col("budget") >= 100_000)
                 # .withColumn("budget_band",
                             # F.when(F.col("budget") < 5e6, "1) < 5M")
                              # .when(F.col("budget") < 30e6, "2) 5-30M")
                              # .when(F.col("budget") < 100e6, "3) 30-100M")
                              # .otherwise("4) >= 100M"))
                 # .groupBy("budget_band")
                 # .agg(F.count("*").alias("n_movies"), *estatisticas("gap_imdb_ml", "gap"),
                      # F.round(F.avg(F.abs("gap_imdb_ml")), 3).alias("avg_abs_gap")))
# gravar_delta(por_orcamento, GOLD, "gap_by_budget", descricao="Diferença IMDb - MovieLens por escalão de orçamento (USD nominais)")

# Gap por filme: base para a lista dos filmes onde as fontes mais divergem
gaps = base.select("movie_id", "tconst", "title", "start_year", "decade", "genres",
                   "imdb_rating", "imdb_votes", "ml_rating_10", "ml_num_ratings",
                   F.round("gap_imdb_ml", 2).alias("gap_imdb_ml"),
                   F.round(F.abs("gap_imdb_ml"), 2).alias("abs_gap"))
gaps = gravar_delta(gaps, GOLD, "movie_rating_gaps", cluster_by=["start_year"],
                    descricao="Diferença entre a nota IMDb e a nota MovieLens (escala 1-10) por filme")

# [TMDB desativada] o orçamento e a receita vêm da TMDB
# # Retorno financeiro (receita / orçamento) por género; orçamento mínimo de 100 mil USD para excluir valores errados
# retorno = (unificada.where((F.col("budget") >= 100_000) & F.col("revenue").isNotNull())
           # .withColumn("genre", F.explode("genres"))
           # .withColumn("roi", F.col("revenue") / F.col("budget"))
           # .groupBy("genre")
           # .agg(F.count("*").alias("n_movies"),
                # F.round(F.percentile_approx("roi", 0.5), 2).alias("median_roi"),
                # F.round(F.percentile_approx("roi", 0.25), 2).alias("p25_roi"),
                # F.round(F.percentile_approx("roi", 0.75), 2).alias("p75_roi"),
                # F.round(100 * F.avg((F.col("roi") > 1).cast("int")), 1).alias("pct_receita_maior_orcamento"),
                # F.round(F.avg("imdb_rating"), 2).alias("avg_imdb_rating"))
           # .where("n_movies >= 10"))
# gravar_delta(retorno, GOLD, "roi_by_genre", descricao="Retorno (receita/orçamento) por género: mediana e quartis, orçamento >= 100k USD")

# Dados não estruturados: tags dos filmes em que as fontes divergem 1 ponto ou mais
grupos = (gaps.withColumn("grupo", F.when(F.col("gap_imdb_ml") >= 1, "IMDb mais generoso")
                                    .when(F.col("gap_imdb_ml") <= -1, "MovieLens mais generoso"))
          .where(F.col("grupo").isNotNull())
          .select("movie_id", "grupo"))
tags_gap = (spark.table(nome_tabela(SILVER, "ml_tags")).join(F.broadcast(grupos), "movie_id")
            .groupBy("grupo", "tag").agg(F.countDistinct("movie_id").alias("n_movies"))
            .where("n_movies >= 3"))
gravar_delta(tags_gap, GOLD, "tags_by_gap",
             descricao="Tags MovieLens mais frequentes nos filmes onde IMDb e MovieLens divergem >= 1 ponto")

### 4.2 Modelo explicativo com Spark ML: que fatores explicam a diferença de notas?

As tabelas por grupo (género, idioma, década) olham para um fator de cada vez. Mas os fatores misturam-se: os filmes antigos têm mais votos, os filmes não ingleses concentram-se em certos géneros. Uma **regressão linear** separa o efeito de cada um: cada coeficiente diz quantos pontos de gap esse fator acrescenta **mantendo os outros constantes**.

**Variável a explicar:** o gap IMDb − MovieLens de cada filme (população `base`).
**Variáveis explicativas:** décadas desde 2000, log10 dos votos IMDb e os 8 géneros mais frequentes (0/1). A variável de idioma não inglês está preservada como extensão opcional.

**Porque mudámos para `LinearRegression` do Spark ML.** A abordagem com pandas/numpy não estava a servir o objetivo de Big Data: embora funcionasse com os poucos milhares de linhas desta tabela, trazia os dados para uma só máquina e não aproveitava o cluster. Por isso passámos a treinar o modelo no cluster, diretamente sobre o DataFrame Spark. O Spark também calcula os erros-padrão e os p-values.

**O que o código faz, passo a passo:**
1. **Prepara as variáveis** no DataFrame do Spark e retira as linhas com valores em falta (`dropna`).
2. **`VectorAssembler`** junta as variáveis explicativas numa só coluna do tipo vetor (`features`), que é o formato que os modelos do Spark ML recebem.
3. **`LinearRegression`** ajusta o modelo por mínimos quadrados. Usamos `solver="normal"` (resolve as equações normais diretamente), porque é o único que dá **erros-padrão, estatísticas t e p-values** no resumo do modelo. `regParam=0.0` desliga a regularização: queremos a regressão clássica, para interpretar os coeficientes.
4. **`modelo.summary`** devolve o R², o nº de filmes e, para cada coeficiente, o erro-padrão, o t e o p-value. Nestas listas, a constante (*intercept*) vem em último lugar.
5. **Grava** os coeficientes em `gold.gap_model_coefficients`, que alimenta a Figura 7.

**Leitura dos p-values.** Um p-value abaixo de 0,05 é evidência contra um coeficiente nulo, condicionada às hipóteses do modelo. Como são testados vários coeficientes, interprete-os como resultados exploratórios; a regressão mostra associações, não causalidade.

**Âmbito do modelo.** Esta regressão é exploratória: procura verificar se a década, o volume de votos e alguns géneros estão associados ao gap IMDb − MovieLens. Não foi construída para prever a nota de cada filme nem para maximizar o R². A nota MovieLens (0,5–5) é multiplicada por 2 para ficar na escala 1–10 do IMDb; usar escalas comparáveis não demonstra que género e nota sejam independentes.

**Objetivo técnico.** O exercício também demonstra o percurso de modelação com Spark ML: preparar as variáveis no DataFrame Spark, ajustar o modelo, guardar coeficientes na Gold e apresentá-los numa figura. É uma demonstração de execução e integração no Databricks, não um benchmark de velocidade. Embora a preparação use 32 milhões de avaliações MovieLens, o ajuste da regressão é feito sobre 26 614 pares de filmes. Assim como, não representa a máxima capacidade analítica do grupo para obtenção de modelos de regressão além do teste inicial exploratório, pois não se entende que esse seja o âmbito do projecto.

**Próximos passos.** Como próximos passos, compararíamos o modelo ajustado com um modelo apenas por género e testaríamos interações de combinações frequentes, como Ação × Comédia, verificando também a colinearidade entre géneros. Avaliaríamos se ponderar as médias pelo número de avaliações MovieLens altera os resultados, dado que médias com poucas avaliações são menos precisas. Por fim, analisaríamos resíduos e observações influentes, usaríamos erros-padrão robustos e controlaríamos os testes múltiplos para avaliar a estabilidade dos resultados.

In [0]:
# =============================================================================
# ETAPA 4.2 — MODELO: regressão linear do gap IMDb - MovieLens com Spark ML
# =============================================================================
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# 1) Variáveis explicativas: os 8 géneros com mais filmes entram como 0/1
generos_top = [linha.genre for linha in spark.table(nome_tabela(GOLD, "gap_by_genre"))
               .orderBy(F.desc("n_movies")).limit(8).collect()]
colunas_genero = [f"genero_{g.replace('-', '_')}" for g in generos_top]
variaveis = ["decadas_desde_2000", "log10_votos_imdb"] + colunas_genero   # [TMDB desativada] "idioma_nao_ingles"

dados_modelo = (base.select(
                    "gap_imdb_ml",
                    ((F.col("decade") - 2000) / 10).alias("decadas_desde_2000"),
                    # [TMDB desativada] 
                    # (F.col("original_language") != "en").cast("int").alias("idioma_nao_ingles"),
                    F.log10("imdb_votes").alias("log10_votos_imdb"),
                    *[F.coalesce(F.array_contains("genres", g), F.lit(False)).cast("int").alias(c)
                      for g, c in zip(generos_top, colunas_genero)])
                .dropna())

# 2) Juntar as variáveis numa coluna vetor 'features' (formato de entrada do Spark ML)
montador = VectorAssembler(inputCols=variaveis, outputCol="features")

# 3) Ajustar a regressão: solver "normal" para ter erros-padrão e p-values, sem regularização
regressao = LinearRegression(featuresCol="features", labelCol="gap_imdb_ml",
                             solver="normal", regParam=0.0, fitIntercept=True)
modelo = regressao.fit(montador.transform(dados_modelo))
resumo = modelo.summary

# 4) Uma linha por coeficiente. Nas listas do resumo a constante vem em último lugar,
#    por isso juntamos o intercept no fim e depois passamo-lo para o início da tabela.
nomes = variaveis + ["(constante)"]
coeficientes = list(modelo.coefficients) + [modelo.intercept]
linhas_modelo = [(nome, float(coef), float(ep), float(t), float(p))
                 for nome, coef, ep, t, p in zip(nomes, coeficientes, resumo.coefficientStandardErrors,
                                                 resumo.tValues, resumo.pValues)]
linhas_modelo = linhas_modelo[-1:] + linhas_modelo[:-1]

n_filmes_modelo = int(resumo.numInstances)
if n_filmes_modelo < 100:
    print(f"Aviso: só {n_filmes_modelo} filmes; o modelo é pouco fiável. Baixar min_votos_imdb ou min_avaliacoes_ml.")

# 5) Gravar na gold
coeficientes_df = (spark.createDataFrame(linhas_modelo,
                                         "variavel string, coeficiente double, erro_padrao double, t double, p_value double")
                   .withColumn("significativo_5pct", F.col("p_value") < 0.05)
                   .withColumn("r2", F.lit(float(resumo.r2)))
                   .withColumn("n", F.lit(n_filmes_modelo)))
gravar_delta(coeficientes_df, GOLD, "gap_model_coefficients",
             descricao="Regressão linear (Spark ML) do gap IMDb - MovieLens: coeficientes, erros-padrão e p-values")
print(f"n = {n_filmes_modelo:,} filmes | R² = {resumo.r2:.3f}")
display(spark.table(nome_tabela(GOLD, "gap_model_coefficients")))

### 4.3 Perguntas descritivas e de benchmarking (comportamento dos utilizadores MovieLens)

| # | Pergunta | Tabela gold | Figura |
|---|---|---|---|
| 1 | Que géneros de filmes existem no conjunto de dados? | `genre_catalog` | 11 |
| 2 | Quantos filmes há por género? | `genre_catalog` | 11 |
| 3 | Como se distribuem as avaliações por utilizador? | `user_activity` | 12 |
| 4 | Quais são os filmes com a classificação média mais alta? | `top_movies` | 13 |
| 5 | Como mudam, ao longo do tempo, o volume, a média e a distribuição dos ratings? | `ratings_over_time` | 14 |
| 6 | Que utilizadores são sinalizados por picos de atividade, avaliações no mesmo segundo ou concentração em poucas notas? | `user_activity` | 15 |
| 7 | Como muda a sinalização ao excluir as primeiras 24 horas após a primeira avaliação observada? | `flag_sensitivity` | 15 |
| 8 | Que filmes registam as maiores subidas ou descidas de classificação média entre períodos? | `movie_rating_shifts` | 16 |
| 9 | Em quantas avaliações se baseiam essas mudanças? | `movie_rating_shifts` (`n_ratings_total`) | 16 |
| 10 | Que filmes recebem, no mesmo dia, uma concentração de notas muito altas ou muito baixas face ao seu histórico? | `movie_day_anomalies` | 17 |
| 11 | Que proporção dos participantes já tinha sido sinalizada por atividade invulgar? | `anomaly_participants` | 17 |

**Regras de sinalização** (constantes na célula seguinte, fáceis de ajustar e justificar). São heurísticas para revisão exploratória, não evidência de manipulação ou coordenação.
* **Pico de atividade:** um dia com pelo menos `PICO_MIN_DIA` avaliações **e** acima do percentil 99 de todos os dias-utilizador.
* **Mesmo segundo:** `MESMO_SEGUNDO_MIN` ou mais avaliações com o mesmo timestamp (difícil para um humano).
* **Concentração em poucas notas:** utilizador com pelo menos `CONC_MIN_AVALIACOES` avaliações em que a nota mais usada representa `CONC_PROPORCAO_TOP` ou mais do total.
* **Sem as primeiras 24 horas:** o conjunto não fornece a hora de registo. Por isso, excluímos as 24 horas seguintes à primeira avaliação observada como análise de sensibilidade; os resultados não identificam esse período como o registo do utilizador.
* **Data da sinalização:** primeiro dia de pico, primeiro segundo suspeito, ou dia em que o utilizador atinge `CONC_MIN_AVALIACOES` avaliações. Serve para saber se alguém "já tinha sido sinalizado" antes de participar numa anomalia.
* **Mudança entre períodos:** média de cada filme por períodos de `ANOS_PERIODO` anos (mínimo `MIN_AVALIACOES_PERIODO` avaliações por período), comparada com o período anterior com dados.
* **Concentração no mesmo dia:** dia com pelo menos `DIA_MIN_AVALIACOES` avaliações de um filme em que a proporção de notas ≥ `NOTA_EXTREMA_ALTA` (ou ≤ `NOTA_EXTREMA_BAIXA`) supera a do resto do histórico do filme em `DIF_PROPORCAO` ou mais, com z ≥ `Z_MIN` (3). Como se testam muitos pares filme-dia, os casos sinalizados são exploratórios e exigem validação.

In [0]:
# =============================================================================
# ETAPA 4.3.0 — Parâmetros das regras e dados de partida (MovieLens)
# =============================================================================
PICO_MIN_DIA           = 50     # avaliações num só dia
MESMO_SEGUNDO_MIN      = 3      # avaliações no mesmo segundo
CONC_MIN_AVALIACOES    = 20     # mínimo de avaliações para avaliar a concentração
CONC_PROPORCAO_TOP     = 0.70   # proporção da nota mais usada
ANOS_PERIODO           = 5      # tamanho de cada período
MIN_AVALIACOES_PERIODO = 10     # avaliações mínimas de um filme por período
NOTA_EXTREMA_ALTA, NOTA_EXTREMA_BAIXA = 4.5, 1.0
DIA_MIN_AVALIACOES     = 5      # avaliações mínimas de um filme num dia
HIST_MIN_AVALIACOES    = 10     # histórico mínimo do filme fora desse dia
DIF_PROPORCAO          = 0.40   # diferença mínima de proporção face ao histórico
Z_MIN                  = 3      # z mínimo para considerar a diferença "invulgar"
VOTOS_FICTICIOS        = 20     # "votos fictícios" da média bayesiana no top de filmes

# Avaliações e filmes do MovieLens (silver), usados em todas as perguntas desta secção
avaliacoes = spark.table(nome_tabela(SILVER, "ml_ratings")).select("user_id", "movie_id", "rating", "rated_at")
filmes = spark.table(nome_tabela(SILVER, "ml_movies")).select("movie_id", "title", "year", "genres")

In [0]:
# =============================================================================
# ETAPA 4.3.1 — Perguntas 1-2: que géneros existem e quantos filmes têm
# =============================================================================
# explode: um filme com 3 géneros passa a 3 linhas (filme, género)
generos_imdb = (spark.table(nome_tabela(SILVER, "movies_unified")).select("movie_id", F.explode("genres").alias("genre"))
                .groupBy("genre").agg(F.countDistinct("movie_id").alias("n_movies_imdb")))
generos_ml = (filmes.select("movie_id", F.explode("genres").alias("genre"))
              .groupBy("genre").agg(F.countDistinct("movie_id").alias("n_movies_movielens")))
# join "full": mantém os géneros que só existem numa das duas fontes (com 0 na outra)
catalogo_generos = generos_imdb.join(generos_ml, "genre", "full").fillna(0, ["n_movies_imdb", "n_movies_movielens"])
gravar_delta(catalogo_generos, GOLD, "genre_catalog", descricao="Géneros existentes e nº de filmes por género (IMDb e MovieLens)")

In [0]:
# =============================================================================
# ETAPA 4.3.2 — Perguntas 3, 6 e 7: atividade e sinalização dos utilizadores
# =============================================================================
def sinalizar_utilizadores(av, sufixo):
    """Aplica as três regras de sinalização a um conjunto de avaliações (colunas user_id, movie_id,
    rating, rated_at) e devolve uma linha por utilizador. O 'sufixo' é acrescentado ao nome das
    colunas (ex.: '_sem_24h') para distinguir as duas versões."""
    av = av.withColumn("rating_date", F.to_date("rated_at"))
    utilizadores = av.groupBy("user_id").agg(F.count("*").alias("n_ratings"))

    # Regra 1, pico de atividade: dias com >= PICO_MIN_DIA avaliações e acima do percentil 99
    por_dia = av.groupBy("user_id", "rating_date").agg(F.count("*").alias("n_day"))
    p99 = por_dia.agg(F.percentile_approx("n_day", 0.99)).first()[0] or 0
    picos = (por_dia.where((F.col("n_day") >= PICO_MIN_DIA) & (F.col("n_day") > p99))
             .groupBy("user_id").agg(F.count("*").alias("spike_days"), F.max("n_day").alias("max_ratings_day"),
                                     F.min("rating_date").alias("spike_date")))

    # Regra 2, mesmo segundo: MESMO_SEGUNDO_MIN ou mais avaliações com o mesmo timestamp
    mesmo_segundo = (av.groupBy("user_id", "rated_at").agg(F.count("*").alias("n_sec"))
                     .where(F.col("n_sec") >= MESMO_SEGUNDO_MIN)
                     .groupBy("user_id").agg(F.count("*").alias("same_second_events"),
                                             F.max("n_sec").alias("max_same_second"),
                                             F.min(F.to_date("rated_at")).alias("same_second_date")))

    # Regra 3, concentração: que proporção das notas do utilizador é a nota que ele mais usa?
    por_utilizador = Window.partitionBy("user_id")   # janela = "todas as linhas do mesmo utilizador"
    concentracao = (av.groupBy("user_id", "rating").count()
                    .withColumn("share", F.col("count") / F.sum("count").over(por_utilizador))
                    .groupBy("user_id").agg(F.max("share").alias("top_share"), F.count("*").alias("distinct_values")))
    # Data em que o utilizador chega à avaliação nº CONC_MIN_AVALIACOES (a partir daí a regra 3 aplica-se)
    n_esima = (av.withColumn("_n", F.row_number().over(por_utilizador.orderBy("rated_at", "movie_id")))
               .where(F.col("_n") == CONC_MIN_AVALIACOES).select("user_id", F.col("rating_date").alias("conc_date")))

    # Juntar tudo (left join: quem não tem picos fica com nulo nessas colunas)
    resultado = (utilizadores.join(picos, "user_id", "left").join(mesmo_segundo, "user_id", "left")
                 .join(concentracao, "user_id", "left").join(n_esima, "user_id", "left")
                 .withColumn("flag_spike", F.col("spike_days").isNotNull())
                 .withColumn("flag_same_second", F.col("same_second_events").isNotNull())
                 .withColumn("flag_concentration", (F.col("n_ratings") >= CONC_MIN_AVALIACOES)
                             & (F.col("top_share") >= CONC_PROPORCAO_TOP))
                 .withColumn("flag_any", F.col("flag_spike") | F.col("flag_same_second") | F.col("flag_concentration")))

    # Data da primeira sinalização = a mais antiga das três datas, contando só as regras que dispararam.
    # F.when(...) sem otherwise dá nulo quando a regra não disparou; F.least ignora os nulos.
    resultado = resultado.withColumn("first_flag_date",
                                     F.least(F.when(F.col("flag_spike"), F.col("spike_date")),
                                             F.when(F.col("flag_same_second"), F.col("same_second_date")),
                                             F.when(F.col("flag_concentration"), F.col("conc_date"))))
    colunas = ["n_ratings", "spike_days", "max_ratings_day", "max_same_second", "top_share", "distinct_values",
               "flag_spike", "flag_same_second", "flag_concentration", "flag_any", "first_flag_date"]
    return resultado.select("user_id", *[F.col(c).alias(c + sufixo) for c in colunas])


# Pergunta 3: estatísticas de cada utilizador
resumo_utilizadores = avaliacoes.groupBy("user_id").agg(
    F.round(F.avg("rating"), 3).alias("avg_rating"), F.round(F.stddev("rating"), 3).alias("sd_rating"),
    F.min("rated_at").alias("first_rating_at"), F.max("rated_at").alias("last_rating_at"),
    F.countDistinct(F.to_date("rated_at")).alias("active_days"))

# Pergunta 7: as mesmas avaliações sem as primeiras 24 horas de cada utilizador
apos_24h = (avaliacoes.join(resumo_utilizadores.select("user_id", "first_rating_at"), "user_id")
            .where(F.col("rated_at") >= F.col("first_rating_at") + F.expr("INTERVAL 24 HOURS"))
            .drop("first_rating_at"))

# Pergunta 6: sinalização com todas as avaliações (sufixo "") e sem as primeiras 24 h
atividade = (resumo_utilizadores.join(sinalizar_utilizadores(avaliacoes, ""), "user_id")
             .join(sinalizar_utilizadores(apos_24h, "_sem_24h"), "user_id", "left")
             .fillna(False, ["flag_spike_sem_24h", "flag_same_second_sem_24h",
                             "flag_concentration_sem_24h", "flag_any_sem_24h"]))
verificar_chave_unica(atividade, "user_id", "gold.user_activity")
atividade = gravar_delta(atividade, GOLD, "user_activity",
                         descricao="Atividade e sinalização por utilizador MovieLens (todas as avaliações e sem as primeiras 24 h)")

# Pergunta 7 (resumo): quantos utilizadores cada regra sinaliza, com e sem as primeiras 24 h.
# Uma só agregação conta todas as regras de uma vez.
regras_sinalizacao = [("flag_spike", "Pico de atividade"), ("flag_same_second", "Avaliações no mesmo segundo"),
                      ("flag_concentration", "Concentração em poucas notas"), ("flag_any", "Qualquer sinalização")]
contagens_flags = atividade.agg(
    F.count("*").alias("total"),
    *[F.sum(F.col(c).cast("int")).alias(c) for c, _ in regras_sinalizacao],
    *[F.sum(F.col(c + "_sem_24h").cast("int")).alias(c + "_sem_24h") for c, _ in regras_sinalizacao]).first()
linhas_sens = [(rotulo, int(contagens_flags[c] or 0), int(contagens_flags[c + "_sem_24h"] or 0))
               for c, rotulo in regras_sinalizacao]
sensibilidade = (spark.createDataFrame(linhas_sens, "sinalizacao string, utilizadores_todas long, utilizadores_sem_24h long")
                 .withColumn("total_utilizadores", F.lit(int(contagens_flags["total"])))
                 .withColumn("variacao_pct", F.round(100 * (F.col("utilizadores_sem_24h") - F.col("utilizadores_todas"))
                                                     / F.greatest(F.col("utilizadores_todas"), F.lit(1)), 1)))
gravar_delta(sensibilidade, GOLD, "flag_sensitivity", descricao="Utilizadores sinalizados com e sem as primeiras 24 horas")

In [0]:
# =============================================================================
# ETAPA 4.3.3 — Pergunta 4: filmes com a classificação média mais alta (média bayesiana)
# =============================================================================
# A média bayesiana junta VOTOS_FICTICIOS "votos" com a média global.
# Um filme com 2 notas de 5 fica perto da média global; um com 500 notas de 4,5 quase não muda.
media_global = avaliacoes.agg(F.avg("rating")).first()[0]
top = (avaliacoes.groupBy("movie_id").agg(F.count("*").alias("n_ratings"), F.avg("rating").alias("avg_rating"))
       .join(filmes, "movie_id", "left")
       .withColumn("bayes_rating", (F.col("n_ratings") * F.col("avg_rating") + VOTOS_FICTICIOS * F.lit(media_global))
                                   / (F.col("n_ratings") + VOTOS_FICTICIOS))
       .withColumn("rank_bayes", F.row_number().over(Window.orderBy(F.desc("bayes_rating"), F.desc("n_ratings"),
                                                                    "movie_id")))
       .select("rank_bayes", "movie_id", "title", "year", "n_ratings",
               F.round("avg_rating", 3).alias("avg_rating"), F.round("bayes_rating", 3).alias("bayes_rating")))
gravar_delta(top, GOLD, "top_movies", descricao="Filmes por nota média MovieLens (simples e bayesiana, que penaliza poucas avaliações)")

In [0]:
# =============================================================================
# ETAPA 4.3.4 — Pergunta 5: evolução mensal do volume, da média e da distribuição das notas
# =============================================================================
notas_possiveis = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
# Para cada nota possível, a média de (rating == v) é a proporção de avaliações com essa nota
colunas_pct = [F.round(F.avg((F.col("rating") == v).cast("int")), 4).alias("pct_" + str(v).replace(".", "_"))
               for v in notas_possiveis]
ao_longo_tempo = (avaliacoes.withColumn("month", F.date_trunc("month", "rated_at").cast("date"))
                  .groupBy("month")
                  .agg(F.count("*").alias("n_ratings"), F.round(F.avg("rating"), 3).alias("avg_rating"),
                       F.round(F.stddev("rating"), 3).alias("sd_rating"),
                       F.countDistinct("user_id").alias("n_users"), *colunas_pct))
gravar_delta(ao_longo_tempo, GOLD, "ratings_over_time", descricao="Volume, média e distribuição das notas por mês")

In [0]:
# =============================================================================
# ETAPA 4.3.5 — Perguntas 8-9: maiores subidas e descidas da nota média entre períodos
# =============================================================================
# Período de 5 anos: 1997 -> 1995, 2003 -> 2000 (arredonda o ano para baixo ao múltiplo de 5)
por_periodo = (avaliacoes.withColumn("period", (F.floor(F.year("rated_at") / ANOS_PERIODO) * ANOS_PERIODO).cast("int"))
               .groupBy("movie_id", "period").agg(F.count("*").alias("n"), F.avg("rating").alias("avg"))
               .where(F.col("n") >= MIN_AVALIACOES_PERIODO))

# lag = valor da linha anterior na janela, ou seja, o período anterior do mesmo filme
por_filme = Window.partitionBy("movie_id").orderBy("period")
variacoes = (por_periodo
             .withColumn("prev_period", F.lag("period").over(por_filme))
             .withColumn("prev_avg", F.lag("avg").over(por_filme))
             .withColumn("prev_n", F.lag("n").over(por_filme))
             .where(F.col("prev_period").isNotNull())          # o primeiro período não tem com que comparar
             .join(filmes.select("movie_id", "title"), "movie_id", "left")
             .select("movie_id", "title",
                     F.concat_ws("-", "prev_period", (F.col("prev_period") + ANOS_PERIODO - 1).cast("string")).alias("periodo_anterior"),
                     F.concat_ws("-", "period", (F.col("period") + ANOS_PERIODO - 1).cast("string")).alias("periodo_atual"),
                     F.round("prev_avg", 3).alias("avg_anterior"), F.round("avg", 3).alias("avg_atual"),
                     F.round(F.col("avg") - F.col("prev_avg"), 3).alias("variacao"),
                     F.col("prev_n").alias("n_anterior"), F.col("n").alias("n_atual"),
                     (F.col("prev_n") + F.col("n")).alias("n_ratings_total")))     # pergunta 9
gravar_delta(variacoes, GOLD, "movie_rating_shifts",
             descricao=f"Variação da nota média de cada filme entre períodos de {ANOS_PERIODO} anos "
                       f"(mín. {MIN_AVALIACOES_PERIODO} avaliações por período)")

In [0]:
# =============================================================================
# ETAPA 4.3.6 — Perguntas 10-11: notas extremas concentradas num só dia
# =============================================================================
def z_proporcao(proporcao_dia, proporcao_hist):
    """Quantos desvios-padrão a proporção do dia está acima da proporção habitual do filme.
    Fórmula do teste a uma proporção: z = (p_dia - p_hist) / raiz(p_hist * (1 - p_hist) / n_dia).
    p_hist é limitado a [0,01; 0,99] para nunca dividir por zero."""
    p = F.greatest(F.least(proporcao_hist, F.lit(0.99)), F.lit(0.01))
    return (proporcao_dia - proporcao_hist) / F.sqrt(p * (1 - p) / F.col("n_day"))


diarias = (avaliacoes.withColumn("rating_date", F.to_date("rated_at"))
           .withColumn("is_high", (F.col("rating") >= NOTA_EXTREMA_ALTA).cast("int"))   # 1 se nota muito alta
           .withColumn("is_low", (F.col("rating") <= NOTA_EXTREMA_BAIXA).cast("int")))  # 1 se nota muito baixa

# 1) Contagens por filme e dia, e totais por filme
filme_dia = diarias.groupBy("movie_id", "rating_date").agg(
    F.count("*").alias("n_day"), F.sum("is_high").alias("high_day"), F.sum("is_low").alias("low_day"))
totais = diarias.groupBy("movie_id").agg(
    F.count("*").alias("n_tot"), F.sum("is_high").alias("high_tot"), F.sum("is_low").alias("low_tot"))

# 2) Pergunta 10: comparar a proporção de notas extremas no dia com a do resto do histórico
#    ("histórico" = todas as avaliações do filme menos as desse dia)
anomalias = (filme_dia.join(totais, "movie_id")
             .withColumn("n_hist", F.col("n_tot") - F.col("n_day"))
             .where((F.col("n_day") >= DIA_MIN_AVALIACOES) & (F.col("n_hist") >= HIST_MIN_AVALIACOES))
             .withColumn("share_high_day", F.col("high_day") / F.col("n_day"))
             .withColumn("share_high_hist", (F.col("high_tot") - F.col("high_day")) / F.col("n_hist"))
             .withColumn("share_low_day", F.col("low_day") / F.col("n_day"))
             .withColumn("share_low_hist", (F.col("low_tot") - F.col("low_day")) / F.col("n_hist"))
             .withColumn("z_high", z_proporcao(F.col("share_high_day"), F.col("share_high_hist")))
             .withColumn("z_low", z_proporcao(F.col("share_low_day"), F.col("share_low_hist"))))
# Anomalia = diferença grande (>= DIF_PROPORCAO) E pouco provável por acaso (z >= Z_MIN)
alta = (F.col("share_high_day") - F.col("share_high_hist") >= DIF_PROPORCAO) & (F.col("z_high") >= Z_MIN)
baixa = (F.col("share_low_day") - F.col("share_low_hist") >= DIF_PROPORCAO) & (F.col("z_low") >= Z_MIN)
anomalias = (anomalias.withColumn("tipo", F.when(alta, "notas muito altas").when(baixa, "notas muito baixas"))
             .where(F.col("tipo").isNotNull()))

# 3) Pergunta 11: quem avaliou esse filme nesse dia, e se já estava sinalizado antes desse dia
sinalizados = spark.table(nome_tabela(GOLD, "user_activity")).select("user_id", "first_flag_date")
participantes = (diarias.join(F.broadcast(anomalias.select("movie_id", "rating_date", "tipo")), ["movie_id", "rating_date"])
                 .join(sinalizados, "user_id", "left")
                 .withColumn("ja_sinalizado", F.coalesce(F.col("first_flag_date") < F.col("rating_date"), F.lit(False))))
por_anomalia = participantes.groupBy("movie_id", "rating_date").agg(
    F.countDistinct("user_id").alias("n_participants"),
    F.countDistinct(F.when(F.col("ja_sinalizado"), F.col("user_id"))).alias("n_ja_sinalizados"))
anomalias_final = (anomalias.join(por_anomalia, ["movie_id", "rating_date"], "left")
                   .join(filmes.select("movie_id", "title"), "movie_id", "left")
                   .select("movie_id", "title", "rating_date", "tipo", "n_day", "n_hist",
                           F.round("share_high_day", 3).alias("share_high_day"), F.round("share_high_hist", 3).alias("share_high_hist"),
                           F.round("share_low_day", 3).alias("share_low_day"), F.round("share_low_hist", 3).alias("share_low_hist"),
                           F.round(F.greatest("z_high", "z_low"), 2).alias("z"), "n_participants", "n_ja_sinalizados"))
gravar_delta(anomalias_final, GOLD, "movie_day_anomalies",
             descricao="Dias com concentração anormal de notas muito altas ou muito baixas num filme, face ao seu histórico")

# 4) Uma linha por participante: já estava sinalizado antes da sua PRIMEIRA participação?
por_participante = (participantes.groupBy("user_id")
                    .agg(F.min("rating_date").alias("primeira_participacao"),
                         F.first("first_flag_date", ignorenulls=True).alias("first_flag_date"),
                         F.countDistinct("movie_id", "rating_date").alias("n_anomalias"))
                    .withColumn("ja_sinalizado", F.col("first_flag_date") < F.col("primeira_participacao"))
                    .fillna(False, ["ja_sinalizado"]))
gravar_delta(por_participante, GOLD, "anomaly_participants",
             descricao="Utilizadores que participaram em anomalias diárias e se já estavam sinalizados antes da primeira participação")

### 4.4 Modelo dimensional: star schema na Gold

As tabelas gold anteriores são **agregados** prontos para cada hipótese. O **star schema** (modelo de Kimball) guarda os mesmos dados **ao detalhe**, organizados para qualquer pergunta futura e para ferramentas de BI (dashboards do Databricks, Power BI):

* **Tabela de factos `fact_rating`**: uma linha por **filme × fonte** (grão). Medidas: nota na escala 1-10 e nº de votos. Pôr a fonte como dimensão permite comparar as fontes incluídas com um simples `GROUP BY`.
* **Dimensões**: `dim_movie` (título, duração e atributos financeiros quando disponíveis), `dim_source` (fonte e escala original) e `dim_year` (ano e década). `dim_language` (idioma original) é uma dimensão opcional.
* **Ponte `bridge_movie_genre`**: um filme tem vários géneros (relação muitos-para-muitos), por isso os géneros ligam-se a `dim_genre` através de uma tabela ponte.

**Filmes sem ano de estreia.** Em vez de ficarem com `year_key` nulo, que violaria a chave estrangeira para `dim_year`, recebem `year_key = -1`, e `dim_year` tem uma linha "ano desconhecido" com essa chave. É a prática habitual de Kimball para membros desconhecidos de uma dimensão.

**Chaves primárias e estrangeiras.** São registadas no Unity Catalog e são **informativas**: documentam o modelo, aparecem no Catalog Explorer e ajudam o otimizador, mas o Databricks não impede duplicados nem chaves órfãs. Por isso a unicidade é verificada com `verificar_chave_unica` antes de cada escrita. Para alterar constraints é preciso ser dono das tabelas; com o catálogo `workspace` quem corre o notebook é sempre o dono. Se o ambiente não as aceitar, o notebook avisa e continua.

A consulta Q6 da secção 6.1 responde à pergunta da Figura 4 diretamente sobre o star schema.

In [0]:
# =============================================================================
# ETAPA 4.4.1 — GOLD · STAR SCHEMA: dimensões, ponte, factos (filme × fonte) e chaves PK/FK
# =============================================================================
ANO_DESCONHECIDO = -1     # year_key dos filmes sem ano de estreia (linha "ano desconhecido" em dim_year)
LINHAS_DIM_FONTE = [(1, "IMDb", "1 a 10", "nota original"),
                    (2, "MovieLens", "0,5 a 5", "nota × 2")]
                    # [TMDB desativada] 
                    # (3, "TMDB", "0 a 10", "nota original")]

# Chaves primárias (tabela -> coluna) e estrangeiras ((tabela, coluna) -> (tabela referenciada, coluna))
CHAVES_PRIMARIAS = {"dim_movie": "movie_key", "dim_source": "source_key", "dim_year": "year_key", "dim_genre": "genre_key"}
                    # [TMDB desativada] 
                    # "dim_language": "language_key", "dim_genre": "genre_key"}
CHAVES_ESTRANGEIRAS = {("fact_rating", "movie_key"): ("dim_movie", "movie_key"),
                       ("fact_rating", "source_key"): ("dim_source", "source_key"),
                       ("fact_rating", "year_key"): ("dim_year", "year_key"),
                       # [TMDB desativada] 
                       # ("fact_rating", "language_key"): ("dim_language", "language_key"),
                       ("bridge_movie_genre", "movie_key"): ("dim_movie", "movie_key"),
                       ("bridge_movie_genre", "genre_key"): ("dim_genre", "genre_key")}


def executar_constraint(instrucao):
    """Executa uma instrução de constraint. As constraints são informativas: se falharem, avisa e continua."""
    try:
        spark.sql(instrucao)
        return True
    except Exception as erro:
        print(f"[aviso] {instrucao[:80]}... -> {type(erro).__name__}")
        return False


# 1) Antes de reescrever as tabelas, retirar as constraints da execução anterior (primeiro FK, depois PK)
for (tabela, coluna) in CHAVES_ESTRANGEIRAS:
    if spark.catalog.tableExists(nome_tabela(GOLD, tabela)):
        executar_constraint(f"ALTER TABLE {nome_tabela(GOLD, tabela)} DROP CONSTRAINT IF EXISTS fk_{tabela}_{coluna}")
for tabela in CHAVES_PRIMARIAS:
    if spark.catalog.tableExists(nome_tabela(GOLD, tabela)):
        executar_constraint(f"ALTER TABLE {nome_tabela(GOLD, tabela)} DROP PRIMARY KEY IF EXISTS CASCADE")

# 2) Dimensões, a partir da silver integrada
filmes_integrados = spark.table(nome_tabela(SILVER, "movies_unified"))

dim_movie = filmes_integrados.select(F.col("movie_id").alias("movie_key"), "tconst", "title", "runtime_minutes")
# [TMDB desativada] colunas da TMDB em dim_movie: "tmdb_id", "budget", "revenue" e o escalão de orçamento:
#     F.when(F.col("budget").isNull(), "desconhecido")
#      .when(F.col("budget") < 5e6, "1) < 5M").when(F.col("budget") < 30e6, "2) 5-30M")
#      .when(F.col("budget") < 100e6, "3) 30-100M").otherwise("4) >= 100M").alias("budget_band")

dim_source = spark.createDataFrame(LINHAS_DIM_FONTE, "source_key int, source_name string, "
                                                     "escala_original string, conversao_1_10 string")

# Anos conhecidos + uma linha para "ano desconhecido" (filmes sem start_year)
dim_year = (filmes_integrados.where(F.col("start_year").isNotNull())
            .select(F.col("start_year").alias("year_key"), "decade").distinct()
            .unionByName(spark.createDataFrame([(ANO_DESCONHECIDO, None)], "year_key int, decade int")))

# [TMDB desativada] o idioma original vem da TMDB
# mapa_idiomas = F.create_map(*[F.lit(x) for par in NOMES_IDIOMAS.items() for x in par])
# dim_language = (filmes_integrados.select(F.coalesce("original_language", F.lit("n/d")).alias("language_key")).distinct()
                # .withColumn("language_name", F.coalesce(mapa_idiomas[F.col("language_key")], F.col("language_key"))))

# Géneros: uma linha por (filme, género); a chave do género é um número sequencial por ordem alfabética
filme_genero = filmes_integrados.select(F.col("movie_id").alias("movie_key"), F.explode("genres").alias("genre_name"))
dim_genre = (filme_genero.select("genre_name").distinct()
             .withColumn("genre_key", F.row_number().over(Window.orderBy("genre_name")))
             .select("genre_key", "genre_name"))
bridge_movie_genre = (filme_genero.join(F.broadcast(dim_genre), "genre_name")
                      .select("movie_key", "genre_key").distinct())

# 3) Factos: cada filme dá até 3 linhas, uma por fonte que tenha nota
chaves = [F.col("movie_id").alias("movie_key"),
          F.coalesce(F.col("start_year"), F.lit(ANO_DESCONHECIDO)).alias("year_key")]
          # [TMDB desativada] F.coalesce("original_language", F.lit("n/d")).alias("language_key")
fact_rating = (
    filmes_integrados.select(*chaves, F.lit(1).alias("source_key"),
                             F.col("imdb_rating").alias("rating_10"), F.col("imdb_votes").alias("n_votes"))
    .unionByName(filmes_integrados.select(*chaves, F.lit(2).alias("source_key"),
                                          F.col("ml_rating_10").alias("rating_10"), F.col("ml_num_ratings").alias("n_votes")))
    # [TMDB desativada]
    # .unionByName(filmes_integrados.select(*chaves, F.lit(3).alias("source_key"),
    #                                       F.col("tmdb_vote_average").alias("rating_10"), F.col("tmdb_vote_count").alias("n_votes")))
    .where(F.col("rating_10").isNotNull()))

# 4) Controlo de qualidade (chaves únicas) e escrita; os factos ficam com Liquid Clustering
tabelas_star = [(dim_movie, "movie_key", "dim_movie", None, "Star schema: dimensão filme (atributos descritivos)"),
                (dim_source, "source_key", "dim_source", None, "Star schema: dimensão fonte da nota"),
                (dim_year, "year_key", "dim_year", None, "Star schema: dimensão ano de estreia e década (-1 = desconhecido)"),
                # [TMDB desativada] 
                # (dim_language, "language_key", "dim_language", None, "Star schema: dimensão idioma original"),
                (dim_genre, "genre_key", "dim_genre", None, "Star schema: dimensão género (IMDb)"),
                (bridge_movie_genre, ["movie_key", "genre_key"], "bridge_movie_genre", None,
                 "Star schema: ponte filme-género (muitos-para-muitos)"),
                (fact_rating, ["movie_key", "source_key"], "fact_rating", ["source_key", "year_key"],
                 "Star schema: factos de avaliação, grão = filme x fonte, nota na escala 1-10")]
for df, chave, tabela, clustering, descricao in tabelas_star:
    verificar_chave_unica(df, chave, f"gold.{tabela}")
    gravar_delta(df, GOLD, tabela, cluster_by=clustering, descricao=descricao)

# 5) Registar as chaves no Unity Catalog: PK (a coluna tem de ser NOT NULL) e depois FK
for tabela, coluna in CHAVES_PRIMARIAS.items():
    if executar_constraint(f"ALTER TABLE {nome_tabela(GOLD, tabela)} ALTER COLUMN {coluna} SET NOT NULL"):
        executar_constraint(f"ALTER TABLE {nome_tabela(GOLD, tabela)} ADD CONSTRAINT pk_{tabela} PRIMARY KEY ({coluna})")
for (tabela, coluna), (referida, coluna_referida) in CHAVES_ESTRANGEIRAS.items():
    executar_constraint(f"ALTER TABLE {nome_tabela(GOLD, tabela)} ADD CONSTRAINT fk_{tabela}_{coluna} "
                        f"FOREIGN KEY ({coluna}) REFERENCES {nome_tabela(GOLD, referida)} ({coluna_referida})")

In [0]:
# =============================================================================
# ETAPA 4.4.2 — Diagrama do star schema e exemplo de consulta
# Desenho em SVG: o que interessa é a imagem, não é preciso perceber o código de desenho.
# =============================================================================
def mostrar_star_schema():
    """Factos ao centro, dimensões à volta, com PK, FK e cardinalidades (1 — N)."""
    L_FACTOS, L_DIM, LINHA, CABECALHO = 230, 200, 21, 30
    VERDE_ESCURO, VERDE, VERDE_CLARO = "#2F6B1A", "#5FA339", "#86B86A"
    VERDE_FUNDO, VERDE_LINHA, VERDE_OLIVA = "#F3F9EE", "#9DC183", "#6B8E23"
    # nome: (x, y, largura, cor, [(coluna, tipo, papel)])   papel: PK, FK, M (medida) ou ""
    tabelas = {
        "fact_rating": (365, 200, L_FACTOS, VERDE_ESCURO, [
            ("movie_key", "int", "PK FK"), ("source_key", "int", "PK FK"), ("year_key", "int", "FK"),
            ("rating_10", "double", "M"), ("n_votes", "bigint", "M")]),
            # [TMDB desativada] 
            # ("language_key", "string", "FK"), ("rating_10", "double", "M"), ("n_votes", "bigint", "M")]),
        "dim_source": (380, 15, L_DIM, VERDE, [
            ("source_key", "int", "PK"), ("source_name", "string", ""), ("escala_original", "string", "")]),
        "dim_year": (730, 80, L_DIM, VERDE, [("year_key", "int", "PK"), ("decade", "int", "")]),
        # [TMDB desativada] 
        # "dim_language": (730, 330, L_DIM, VERDE, [
            # ("language_key", "string", "PK"), ("language_name", "string", "")]),
        "dim_movie": (30, 175, L_DIM, VERDE, [
            ("movie_key", "int", "PK"), ("tconst", "string", ""), ("title", "string", ""),
            ("runtime_minutes", "int", "")]),
            # [TMDB desativada] ("tmdb_id", "int", ""), ("budget", "bigint", ""), ("revenue", "bigint", ""), ("budget_band", "string", "")
        "bridge_movie_genre": (260, 470, L_DIM, VERDE_CLARO, [
            ("movie_key", "int", "PK FK"), ("genre_key", "int", "PK FK")]),
        "dim_genre": (530, 470, L_DIM, VERDE, [("genre_key", "int", "PK"), ("genre_name", "string", "")]),
    }
    ligacoes = [("fact_rating", "dim_source"), ("fact_rating", "dim_year"),   # [TMDB desativada] ("fact_rating", "dim_language")
                ("fact_rating", "dim_movie"), ("bridge_movie_genre", "dim_movie"), ("bridge_movie_genre", "dim_genre")]

    def geometria(nome):
        x, y, w, _, colunas = tabelas[nome]
        return x, y, w, CABECALHO + LINHA * len(colunas) + 6

    def centro(nome):
        x, y, w, h = geometria(nome)
        return x + w / 2, y + h / 2

    def ponto_saida(nome, destino, folga=16):
        """Ponto onde a linha sai da caixa, um pouco para fora (onde fica a cardinalidade)."""
        x, y, w, h = geometria(nome)
        (cx, cy), (tx, ty) = centro(nome), destino
        dx, dy = tx - cx, ty - cy
        t = min((w / 2) / abs(dx) if dx else float("inf"), (h / 2) / abs(dy) if dy else float("inf"))
        norma = (dx * dx + dy * dy) ** 0.5
        return cx + dx * t + dx / norma * folga, cy + dy * t + dy / norma * folga

    svg, rotulos = [], []
    for muitos, um in ligacoes:   # linhas primeiro, para ficarem por baixo das caixas
        (x1, y1), (x2, y2) = centro(muitos), centro(um)
        svg.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{VERDE_LINHA}" stroke-width="2.5"/>')
        for nome, outro, rotulo in ((muitos, (x2, y2), "N"), (um, (x1, y1), "1")):
            lx, ly = ponto_saida(nome, outro)
            rotulos.append(f'<circle cx="{lx}" cy="{ly}" r="10" fill="#fff" stroke="{VERDE}"/>'
                           f'<text x="{lx}" y="{ly + 4}" text-anchor="middle" font-size="11" font-weight="700" '
                           f'fill="{VERDE_ESCURO}">{rotulo}</text>')
    for nome, (x, y, w, cor, colunas) in tabelas.items():
        _, _, _, h = geometria(nome)
        e_factos = nome.startswith("fact")
        svg.append(f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="8" fill="{VERDE_FUNDO}" stroke="{cor}" '
                   f'stroke-width="{3 if e_factos else 2}"/>'
                   f'<path d="M{x},{y + 8} a8,8 0 0 1 8,-8 h{w - 16} a8,8 0 0 1 8,8 v{CABECALHO - 8} h-{w} z" fill="{cor}"/>'
                   f'<text x="{x + 10}" y="{y + 20}" font-size="14" font-weight="700" fill="#fff">'
                   f'{"★ " if e_factos else ""}{nome}</text>')
        for i, (coluna, tipo, papel) in enumerate(colunas):
            ty = y + CABECALHO + 16 + i * LINHA
            peso = "700" if "PK" in papel else "400"
            cor_papel = {"PK": VERDE_ESCURO, "FK": VERDE, "M": VERDE_OLIVA}
            etiquetas = "".join(f'<tspan fill="{cor_papel[p]}" font-weight="700">{p} </tspan>' for p in papel.split())
            svg.append(f'<text x="{x + 10}" y="{ty}" font-size="12.5" font-weight="{peso}" fill="#222" '
                       f'font-family="Consolas,monospace">{coluna}</text>'
                       f'<text x="{x + w - 10}" y="{ty}" font-size="11" text-anchor="end" fill="#777">'
                       f'{etiquetas}{tipo}</text>')
    legenda = ('<text x="30" y="585" font-size="12" fill="#444">'
               f'<tspan fill="{VERDE_ESCURO}" font-weight="700">PK</tspan> chave primária · '
               f'<tspan fill="{VERDE}" font-weight="700">FK</tspan> chave estrangeira · '
               f'<tspan fill="{VERDE_OLIVA}" font-weight="700">M</tspan> medida · '
               '1 — N cardinalidade · grão de fact_rating: um filme × uma fonte</text>')
    displayHTML(f"""
    <div style="background:#fff;color:#222;font-family:Segoe UI,Helvetica,Arial,sans-serif;padding:10px">
      <div style="font-size:18px;font-weight:700">Star schema da camada Gold</div>
      <svg viewBox="0 0 960 600" width="100%" style="max-width:960px" xmlns="http://www.w3.org/2000/svg"
           font-family="Segoe UI,Helvetica,Arial,sans-serif">{''.join(svg + rotulos)}{legenda}</svg>
    </div>""")


mostrar_star_schema()

# Exemplo: a mesma pergunta da Figura 4 (nota média por década e fonte), respondida sobre o star schema
display(spark.sql(f"""
    SELECT y.decade, s.source_name,
           COUNT(*)                   AS n_movies,
           ROUND(AVG(f.rating_10), 2) AS avg_rating_10
    FROM {nome_tabela(GOLD, 'fact_rating')} f
    JOIN {nome_tabela(GOLD, 'dim_source')} s ON f.source_key = s.source_key
    JOIN {nome_tabela(GOLD, 'dim_year')}   y ON f.year_key   = y.year_key
    WHERE y.year_key <> {ANO_DESCONHECIDO}
    GROUP BY y.decade, s.source_name
    ORDER BY y.decade, s.source_name"""))

## 5. Evaluation: análise e gráficos (CRISP-DM fase 5)

Todos os gráficos leem tabelas gold (ou a silver integrada já filtrada), por isso o `toPandas()` só traz poucas linhas para o driver. Cada gráfico tem por baixo um resumo numérico, para ajudar a escrever as conclusões.

Nas comparações de médias por filme, os ICs são aproximações a 95%: média ± 1,96 × desvio-padrão / √n. No modelo, os intervalos dos coeficientes são coeficiente ± 1,96 × erro-padrão. As comparações por género/década e os vários coeficientes não têm correção para múltiplos testes; interprete ICs e p-values como evidência exploratória, junto com o tamanho do efeito e o número de filmes.

In [0]:
# =============================================================================
# ETAPA 5.0 — Função comum aos gráficos
# =============================================================================
def ic95(desvio_padrao, n):
    """Meia largura do intervalo de confiança a 95% para uma média."""
    return 1.96 * desvio_padrao / np.sqrt(n)

### 5.1 Quantos filmes sobrevivem a cada passo de integração?
Ajuda a explicar o tamanho das amostras e os limites das conclusões.

In [0]:
# =============================================================================
# ETAPA 5.1 — Figura 2: funil de integração do IMDb e do MovieLens ([TMDB desativada]: passos da TMDB comentados)
# =============================================================================
unificada, base = populacoes_analise()
funil = [
    ("Filmes IMDb (silver)", spark.table(nome_tabela(SILVER, "imdb_movies")).count()),
    ("Ligados ao MovieLens", unificada.count()),
    ("Com avaliações MovieLens", unificada.where("ml_num_ratings IS NOT NULL").count()),
    # [TMDB desativada] 
    # ("Com detalhe TMDB", unificada.where("in_tmdb").count()),
    (f"base (≥{MIN_VOTOS_IMDB} votos IMDb, ≥{MIN_AVALIACOES_ML} aval. ML)", base.count()),
    # [TMDB desativada] 
    # (f"base_cmp (+ ≥{MIN_VOTOS_TMDB} votos TMDB)", base_cmp.count()),
]
rotulos, valores = zip(*funil)
fig, ax = plt.subplots(figsize=(10, 4))
barras = ax.barh(rotulos[::-1], valores[::-1], color="#4c72b0")
ax.set_xscale("log")
ax.bar_label(barras, labels=[f"{v:,}" for v in valores[::-1]], padding=3, fontsize=9)
ax.set_xlabel("Nº de filmes (escala logarítmica)")
ax.set_title("Figura 2. Funil de integração do IMDb e do MovieLens")
fig.tight_layout()
plt.show()

### 5.2 H1: o IMDb e o MovieLens dão notas diferentes aos mesmos filmes?


In [0]:
# =============================================================================
# ETAPA 5.2 — Figura 3: distribuição das notas e IMDb vs MovieLens (H1)
# =============================================================================
notas = base.select("imdb_rating", "ml_rating_10").toPandas()   # [TMDB desativada] "tmdb_vote_average"

fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))
classes = np.arange(1, 10.25, 0.25)
for coluna, fonte in [("imdb_rating", "IMDb"), ("ml_rating_10", "MovieLens")]:   # [TMDB desativada] ("tmdb_vote_average", "TMDB")
    eixos[0].hist(notas[coluna].dropna(), bins=classes, alpha=0.45, label=f"{fonte} (média {notas[coluna].mean():.2f})",
                  color=CORES_FONTES[fonte])
eixos[0].set_title("Figura 3a. Distribuição das notas (mesmos filmes)")
eixos[0].set_xlabel("Nota (escala 1-10)")
eixos[0].set_ylabel("Nº de filmes")
eixos[0].legend()

eixos[1].scatter(notas["ml_rating_10"], notas["imdb_rating"], s=6, alpha=0.3, color="#555")
eixos[1].plot([1, 10], [1, 10], "r--", lw=1, label="notas iguais (y = x)")
eixos[1].set_xlim(1, 10)
eixos[1].set_ylim(1, 10)
eixos[1].set_xlabel("Nota MovieLens (×2)")
eixos[1].set_ylabel("Nota IMDb")
eixos[1].set_title(f"Figura 3b. IMDb vs MovieLens (r = {notas['imdb_rating'].corr(notas['ml_rating_10']):.2f})")
eixos[1].legend(loc="upper left")
fig.tight_layout()
plt.show()

diferenca = notas["imdb_rating"] - notas["ml_rating_10"]
meia_largura = ic95(diferenca.std(), len(diferenca))
print(f"Filmes comparados: {len(notas):,}")
print(f"Média IMDb {notas['imdb_rating'].mean():.2f} | MovieLens {notas['ml_rating_10'].mean():.2f}")
# [TMDB desativada] print(f"TMDB {notas['tmdb_vote_average'].mean():.2f}")
print(f"Diferença média IMDb − MovieLens: {diferenca.mean():+.2f} pontos "
      f"(IC 95%: {diferenca.mean() - meia_largura:+.2f} a {diferenca.mean() + meia_largura:+.2f})")
print("Se o IC aproximado não contém 0, há evidência de diferença média nesta amostra de filmes (H1).")

### 5.3 H5: as notas mudam com a década?
As bandas sombreadas são intervalos de confiança a 95%. As barras cinzentas mostram quantos filmes há em cada década: décadas com poucos filmes têm bandas largas.

In [0]:
# =============================================================================
# ETAPA 5.3 — Figura 4: nota média por década e fonte (H5)
# =============================================================================
decadas = spark.table(nome_tabela(GOLD, "ratings_by_decade")).orderBy("decade").toPandas()
fig, ax = plt.subplots(figsize=(11, 4.8))
ax2 = ax.twinx()                                    # segundo eixo vertical, para o nº de filmes
ax2.bar(decadas["decade"], decadas["n_movies"], width=7, color="#dddddd", zorder=0)
ax2.set_ylabel("Nº de filmes", color="#888")
ax2.grid(False)
ax.set_zorder(ax2.get_zorder() + 1)                 # linhas por cima das barras
ax.patch.set_visible(False)
for prefixo, fonte in [("imdb", "IMDb"), ("ml", "MovieLens")]:   # [TMDB desativada] ("tmdb", "TMDB")
    media, meia = decadas[f"avg_{prefixo}"], ic95(decadas[f"sd_{prefixo}"], decadas[f"n_{prefixo}"])
    ax.plot(decadas["decade"], media, marker="o", label=fonte, color=CORES_FONTES[fonte])
    ax.fill_between(decadas["decade"], media - meia, media + meia, alpha=0.15, color=CORES_FONTES[fonte])
ax.set_xlabel("Década de estreia")
ax.set_ylabel("Nota média (1-10)")
ax.set_title("Figura 4. Nota média por década e fonte (mesmos filmes, IC 95%)")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

antigos, recentes = decadas[decadas["decade"] < 1980], decadas[decadas["decade"] >= 2000]
if len(antigos) and len(recentes):
    media_antiga = np.average(antigos['avg_imdb'], weights=antigos['n_movies'])
    media_recente = np.average(recentes['avg_imdb'], weights=recentes['n_movies'])
    print(f"Nota IMDb média antes de 1980: {media_antiga:.2f} | desde 2000: {media_recente:.2f}")
    if media_antiga > media_recente:
        print("A diferença descritiva aponta na direção de H5; não demonstra, por si só, um efeito de sobrevivência.")
    else:
        print("Esta comparação descritiva não apoia a previsão de H5 para estes grupos de décadas.")

### 5.4 H2: a diferença IMDb − MovieLens depende do género?
Barras à direita de zero: o IMDb tem média mais alta do que o MovieLens. As linhas mostram ICs aproximados a 95% por género; são comparações individuais sem correção para os vários géneros e devem ser lidas como exploratórias.

In [0]:
# =============================================================================
# ETAPA 5.4 — Figura 5: diferença de nota por género (H2)
# =============================================================================
generos = spark.table(nome_tabela(GOLD, "gap_by_genre")).orderBy("avg_gap").toPandas()
erro = ic95(generos["sd_gap"], generos["n_gap"])
cores = ["#c44e52" if v > 0 else "#4c72b0" for v in generos["avg_gap"]]
fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(generos))))
ax.barh(generos["genre"] + "  (n=" + generos["n_movies"].astype(str) + ")", generos["avg_gap"], xerr=erro,
        color=cores, error_kw={"elinewidth": 1})
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Diferença média IMDb − MovieLens (pontos, escala 1-10)")
ax.set_title("Figura 5. Diferença de nota por género (IC 95%)")
fig.tight_layout()
plt.show()
maior, menor = generos.iloc[-1], generos.iloc[0]
print(f"Maior gap positivo: {maior['genre']} ({maior['avg_gap']:+.3f}) | menor: {menor['genre']} ({menor['avg_gap']:+.3f})")

### 5.5 H3 e H4: idioma e orçamento
_A figura está preservada para apresentar os resultados quando as tabelas correspondentes estiverem disponíveis._

In [0]:
# =============================================================================
# ETAPA 5.5 — Figura 6: diferença por idioma original (H3) e por orçamento (H4)  [TMDB desativada]
# =============================================================================
# Não corre: as tabelas gap_by_language e gap_by_budget dependem da TMDB.
# Para voltar a ativar: retirar o '# ' do início de cada linha.

# idiomas = spark.table(nome_tabela(GOLD, "gap_by_language")).orderBy(F.desc("n_movies")).limit(10).toPandas()
# idiomas["nome"] = idiomas["original_language"].map(NOMES_IDIOMAS).fillna(idiomas["original_language"])
# orcamentos = spark.table(nome_tabela(GOLD, "gap_by_budget")).orderBy("budget_band").toPandas()

# fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5))
# x = np.arange(len(idiomas))
# eixos[0].bar(x - 0.2, idiomas["avg_gap_ml"], 0.4, yerr=ic95(idiomas["sd_gap_ml"], idiomas["n_gap_ml"]),
             # label="IMDb − MovieLens", color=CORES_FONTES["MovieLens"])
# eixos[0].bar(x + 0.2, idiomas["avg_gap_tmdb"], 0.4, yerr=ic95(idiomas["sd_gap_tmdb"], idiomas["n_gap_tmdb"]),
             # label="IMDb − TMDB", color=CORES_FONTES["TMDB"])
# eixos[0].set_xticks(x, idiomas["nome"] + "\n(n=" + idiomas["n_movies"].astype(str) + ")", rotation=45, ha="right", fontsize=8)
# eixos[0].axhline(0, color="black", lw=0.8)
# eixos[0].set_ylabel("Diferença média (pontos)")
# eixos[0].set_title("Figura 6a. Diferença de nota por idioma original (IC 95%)")
# eixos[0].legend()

# orcamentos["budget_band"] = orcamentos["budget_band"] + "\n(n=" + orcamentos["n_movies"].astype(str) + ")"
# eixos[1].bar(orcamentos["budget_band"], orcamentos["avg_gap"], yerr=ic95(orcamentos["sd_gap"], orcamentos["n_gap"]),
             # color="#8172b2", label="diferença média")
# eixos[1].plot(orcamentos["budget_band"], orcamentos["avg_abs_gap"], "ko--", label="diferença absoluta média")
# eixos[1].axhline(0, color="black", lw=0.8)
# eixos[1].set_xlabel("Orçamento (USD nominais)")
# eixos[1].set_ylabel("Pontos")
# eixos[1].set_title("Figura 6b. Diferença IMDb − MovieLens por orçamento")
# eixos[1].legend()
# fig.tight_layout()
# plt.show()
# print("H4: se a diferença absoluta média desce com o orçamento, os filmes caros têm notas mais consensuais.")

### 5.6 Modelo: o efeito de cada fator, com os outros constantes
Cada ponto é um coeficiente da regressão da secção 4.2 e a linha é o seu intervalo de confiança a 95%. Pontos vermelhos são significativos a 5% (o intervalo não cruza o zero).

O R² de 0,037 indica que os preditores incluídos explicam uma parcela pequena da variação dos gaps entre filmes. Isso limita o poder explicativo global do modelo, mas não significa que todos os coeficientes sejam nulos. Os coeficientes de género são associações condicionais; por exemplo, um coeficiente negativo reduz o gap IMDb − MovieLens, não necessariamente a nota total do filme. 

Os p-values baixos são evidência dentro das hipóteses do modelo, não prova definitiva à pergunta central; as comparações são exploratórias, no máximo. Por exemplo, o coeficiente de Horror é de −0,135 pontos. Isso significa um gap IMDb−MovieLens menor, condicionado às outras variáveis do modelo e não que as notas de Horror sejam, em absoluto, mais baixas.

In [0]:
# =============================================================================
# ETAPA 5.6 — Figura 7: coeficientes da regressão (Spark ML)
# =============================================================================
coefs = (spark.table(nome_tabela(GOLD, "gap_model_coefficients")).where("variavel != '(constante)'")
         .toPandas().sort_values("coeficiente"))
r2, n = coefs["r2"].iloc[0], coefs["n"].iloc[0]
fig, ax = plt.subplots(figsize=(9, 0.45 * len(coefs) + 1.5))
ax.errorbar(coefs["coeficiente"], coefs["variavel"], xerr=1.96 * coefs["erro_padrao"], fmt="none",
            ecolor="#999", capsize=3)
ax.scatter(coefs["coeficiente"], coefs["variavel"],
           color=np.where(coefs["significativo_5pct"], "#c44e52", "#999"), zorder=3)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Efeito no gap IMDb − MovieLens (pontos)")
ax.set_title(f"Figura 7. Coeficientes da regressão (n = {n:,}, R² = {r2:.2f})")
fig.tight_layout()
plt.show()
for _, linha in coefs[coefs["significativo_5pct"]].iterrows():
    p = linha["p_value"]
    p_formatado = "< 0.001" if p < 0.001 else f"= {p:.3g}"
    print(f"  {linha['variavel']}: {linha['coeficiente']:+.3f} pontos (p {p_formatado})")
print(f"O modelo explica {100 * r2:.1f}% da variação do gap; o resto depende de fatores não observados.")

### 5.7 Dados não estruturados: que tags aparecem quando as fontes discordam?

As tags comuns: murder, independent film, woman director, revenge e love; aparecem nos dois grupos de filmes com divergência entre as fontes. Isto aponta para temas recorrentes nas tags dos utilizadores, mas não explica por que razão as notas diferem porque existem em ambos os datasets como mais generoso.

In [0]:
# =============================================================================
# ETAPA 5.7 — Figura 8: tags mais frequentes nos filmes com diferença ≥ 1 ponto
# =============================================================================
tags = spark.table(nome_tabela(GOLD, "tags_by_gap")).toPandas()
fig, eixos = plt.subplots(1, 2, figsize=(12, 5))
for ax, (grupo, cor) in zip(eixos, [("IMDb mais generoso", "#c44e52"), ("MovieLens mais generoso", "#4c72b0")]):
    mais_frequentes = tags[tags["grupo"] == grupo].nlargest(12, "n_movies").sort_values("n_movies")
    ax.barh(mais_frequentes["tag"], mais_frequentes["n_movies"], color=cor)
    ax.set_title(grupo)
    ax.set_xlabel("Nº de filmes com a tag")
fig.suptitle("Figura 8. Tags mais frequentes nos filmes com diferença ≥ 1 ponto")
fig.tight_layout()
plt.show()

### 5.8 Retorno financeiro por género
_A figura está preservada para apresentar os resultados quando a tabela correspondente estiver disponível._
A barra é a mediana de receita/orçamento; a linha vai do 1.º ao 3.º quartil. A mediana é usada porque a média é dominada por poucos sucessos enormes.

In [0]:
# =============================================================================
# ETAPA 5.8 — Figura 9: retorno financeiro por género  [TMDB desativada]
# =============================================================================
# Não corre: a tabela roi_by_genre depende do orçamento e da receita da TMDB.
# Para voltar a ativar: retirar o '# ' do início de cada linha.

# retorno = spark.table(nome_tabela(GOLD, "roi_by_genre")).orderBy("median_roi").toPandas()
# fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(retorno))))
# ax.barh(retorno["genre"], retorno["median_roi"], color="#55a868",
        # xerr=[retorno["median_roi"] - retorno["p25_roi"], retorno["p75_roi"] - retorno["median_roi"]],
        # error_kw={"elinewidth": 1, "ecolor": "#333"})
# ax.axvline(1, color="red", ls="--", lw=1, label="receita = orçamento")
# ax.set_xlabel("Receita / orçamento (mediana, com intervalo interquartil)")
# ax.set_title("Figura 9. Retorno financeiro por género (orçamento ≥ 100k USD)")
# ax.legend()
# fig.tight_layout()
# plt.show()

### 5.9 Qualidade dos dados ao longo das execuções

In [0]:
# =============================================================================
# ETAPA 5.9 — Figura 10: controlo de qualidade da silver (última verificação de cada regra)
# =============================================================================
ultima = Window.partitionBy("tabela", "regra").orderBy(F.desc("_checked_at"))
qualidade = (spark.table(nome_tabela(GOLD, "dq_results"))
             .withColumn("_n", F.row_number().over(ultima)).where("_n = 1")
             .orderBy("pct_violacoes").toPandas())
fig, ax = plt.subplots(figsize=(10, 4.5))
barras = ax.barh(qualidade["regra"] + "  [" + qualidade["tabela"] + "]", qualidade["pct_violacoes"],
                 color=np.where(qualidade["pct_violacoes"] > 5, "#c44e52", "#55a868"))
ax.bar_label(barras, labels=[f"{v:.1f}%" for v in qualidade["pct_violacoes"]], padding=3, fontsize=8)
ax.set_xlabel("% de linhas que violam a regra (última verificação)")
ax.set_title("Figura 10. Controlo de qualidade da silver")
fig.tight_layout()
plt.show()

### 5.10 Evidências técnicas (Spark e Delta Lake)

* **Lazy evaluation e plano de execução:** o `explain` mostra o plano otimizado antes de qualquer ação; procurar `BroadcastHashJoin` e `AdaptiveSparkPlan`.
* **Liquid Clustering:** coluna `clusteringColumns` no `DESCRIBE DETAIL`.
* **Delta Lake, time travel e streaming:** o `DESCRIBE HISTORY` da bronze mostra uma operação `STREAMING UPDATE` por cada execução do Auto Loader; qualquer versão pode ser lida com `VERSION AS OF` ou recuperada com `RESTORE`.

In [0]:
# =============================================================================
# ETAPA 5.10 — Plano de execução, detalhe da tabela e histórico Delta
# =============================================================================
(spark.table(nome_tabela(SILVER, "movies_unified"))
      .join(F.broadcast(spark.table(nome_tabela(SILVER, "ml_movies")).select("movie_id", F.col("year").alias("ml_year"))),
            "movie_id")
      .where(F.col("decade") == 1990)
      .groupBy("decade").agg(F.avg("imdb_rating").alias("avg_imdb_rating"))
      .explain(mode="formatted"))
display(spark.sql(f"DESCRIBE DETAIL {nome_tabela(SILVER, 'movies_unified')}")
        .select("name", "format", "numFiles", "sizeInBytes", "clusteringColumns"))
display(spark.sql(f"DESCRIBE HISTORY {nome_tabela(BRONZE, 'ml_ratings')}")
        .select("version", "timestamp", "operation", "operationMetrics"))

### 5.11 Organização física das tabelas: Liquid Clustering em vez de particionamento

**Particionamento tradicional** (`PARTITIONED BY` / `.partitionBy()`) divide a tabela em pastas por valor de uma coluna (ex.: uma pasta por ano). Funciona bem em tabelas muito grandes (centenas de GB ou mais) com colunas de baixa cardinalidade, mas em tabelas pequenas cria **muitos ficheiros pequenos** e torna as leituras mais lentas. Além disso, mudar a coluna de partição obriga a reescrever a tabela inteira.

**Liquid Clustering** (`CLUSTER BY`), **substitui o particionamento e o Z-ORDER** nas tabelas Delta: organiza os ficheiros pelas colunas escolhidas, permite filtrar sem ler ficheiros desnecessários (*data skipping*) e as colunas podem ser mudadas com `ALTER TABLE ... CLUSTER BY` **sem reescrever os dados**. Por isso é a opção usada neste projeto. As duas técnicas não podem ser usadas na mesma tabela.

| Tabela | Colunas de clustering | Porquê |
|---|---|---|
| `silver.imdb_movies` | `start_year` | filtros e agregações por ano e década |
| `silver.ml_ratings` | `movie_id` | agregação das avaliações por filme |
| `silver.movies_unified` | `decade`, `start_year` | todas as análises gold agrupam por década |
| `gold.movie_rating_gaps` | `start_year` | consultas por período |
| `gold.fact_rating` | `source_key`, `year_key` | consultas do star schema filtram por fonte e por ano |

As tabelas bronze só recebem linhas em append (Auto Loader), e as restantes tabelas gold, incluindo as dimensões do star schema, são pequenas: nenhuma precisa de organização física.

Para comparação, o particionamento tradicional seria (não executado, porque as tabelas são pequenas):
```python
df.write.format("delta").partitionBy("decade").saveAsTable("workspace.silver.movies_unified")
```

A célula seguinte corre `OPTIMIZE`, que aplica o clustering aos dados já escritos, e mostra a organização de cada tabela.

In [0]:
# =============================================================================
# ETAPA 5.11 — OPTIMIZE nas tabelas com clustering e organização física de todas as tabelas
# =============================================================================
todas_tabelas = ([(BRONZE, t) for tabelas in TABELAS_BRONZE.values() for t in tabelas]
                 + [(SILVER, t) for tabelas in TABELAS_SILVER.values() for t in tabelas] + [(SILVER, "movies_unified")]
                 + [(GOLD, t) for t in TABELAS_GOLD + TABELAS_STAR])
organizacao = []
for camada, tabela in todas_tabelas:
    destino = nome_tabela(camada, tabela)
    detalhe = spark.sql(f"DESCRIBE DETAIL {destino}").first().asDict()
    clustering = detalhe.get("clusteringColumns") or []
    if clustering:
        spark.sql(f"OPTIMIZE {destino}")   # agrupa os ficheiros pelas colunas de clustering
        detalhe = spark.sql(f"DESCRIBE DETAIL {destino}").first().asDict()
    organizacao.append((camada, tabela, ", ".join(clustering) or "-",
                        ", ".join(detalhe.get("partitionColumns") or []) or "-",
                        int(detalhe.get("numFiles") or 0), round((detalhe.get("sizeInBytes") or 0) / 1024 / 1024, 2)))
display(spark.createDataFrame(organizacao, "camada string, tabela string, clustering string, particoes string, "
                                           "ficheiros long, tamanho_mb double"))

### 5.12 Respostas às perguntas descritivas e de benchmarking
Cada figura responde a uma ou mais perguntas da tabela da secção 4.3.

In [0]:
# =============================================================================
# ETAPA 5.12.1 — Figura 11 (perguntas 1 e 2): géneros e nº de filmes por género
# =============================================================================
catalogo = spark.table(nome_tabela(GOLD, "genre_catalog")).toPandas().sort_values("n_movies_movielens")
fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(catalogo))))
y = np.arange(len(catalogo))
ax.barh(y - 0.2, catalogo["n_movies_movielens"], 0.4, label="MovieLens", color=CORES_FONTES["MovieLens"])
ax.barh(y + 0.2, catalogo["n_movies_imdb"], 0.4, label="IMDb (filmes integrados)", color=CORES_FONTES["IMDb"])
ax.set_yticks(y, catalogo["genre"])
ax.set_xlabel("Nº de filmes (um filme pode ter vários géneros)")
ax.set_title(f"Figura 11. {len(catalogo)} géneros e nº de filmes por género")
ax.legend()
fig.tight_layout()
plt.show()

In [0]:
# =============================================================================
# ETAPA 5.12.2 — Figura 12 (pergunta 3): distribuição das avaliações por utilizador
# =============================================================================
utilizadores = spark.table(nome_tabela(GOLD, "user_activity")).select("n_ratings", "avg_rating", "active_days").toPandas()
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.2))
eixos[0].hist(utilizadores["n_ratings"], bins=np.logspace(np.log10(max(utilizadores["n_ratings"].min(), 1)),
                                                          np.log10(utilizadores["n_ratings"].max() + 1), 30),
              color="#4c72b0")
eixos[0].set_xscale("log")
eixos[0].axvline(utilizadores["n_ratings"].median(), color="red", ls="--",
                 label=f"mediana = {utilizadores['n_ratings'].median():.0f}")
eixos[0].set_xlabel("Avaliações por utilizador (escala log)")
eixos[0].set_ylabel("Nº de utilizadores")
eixos[0].set_title("Figura 12a. Quantas avaliações faz cada utilizador")
eixos[0].legend()
eixos[1].hist(utilizadores["avg_rating"], bins=np.arange(0.5, 5.25, 0.125), color=CORES_FONTES["MovieLens"])
eixos[1].set_xlabel("Nota média do utilizador (0.5 a 5)")
eixos[1].set_title("Figura 12b. Nota média por utilizador")
fig.tight_layout()
plt.show()
ordenados = utilizadores["n_ratings"].sort_values(ascending=False)
print(f"{len(utilizadores):,} utilizadores | mediana {utilizadores['n_ratings'].median():.0f} avaliações | "
      f"os 10% mais ativos fazem {100 * ordenados.head(max(1, len(utilizadores) // 10)).sum() / ordenados.sum():.0f}% "
      "das avaliações")

In [0]:
# =============================================================================
# ETAPA 5.12.3 — Figura 13 (pergunta 4): top 15 filmes por nota média bayesiana
# =============================================================================
top15 = spark.table(nome_tabela(GOLD, "top_movies")).where("rank_bayes <= 15").orderBy(F.desc("rank_bayes")).toPandas()
fig, ax = plt.subplots(figsize=(10, 5.5))
barras = ax.barh(top15["title"].str.slice(0, 45), top15["bayes_rating"], color="#55a868")
ax.bar_label(barras, labels=[f"{b:.2f}  (n={n})" for b, n in zip(top15["bayes_rating"], top15["n_ratings"])],
             padding=3, fontsize=8)
ax.set_xlim(top15["bayes_rating"].min() - 0.3, 5.3)
ax.set_xlabel("Nota média bayesiana (0.5 a 5)")
ax.set_title("Figura 13. Top 15 filmes por nota média (bayesiana, penaliza poucas avaliações)")
fig.tight_layout()
plt.show()

In [0]:
# =============================================================================
# ETAPA 5.12.4 — Figura 14 (pergunta 5): volume, média e distribuição das notas ao longo do tempo
# =============================================================================
mensal = spark.table(nome_tabela(GOLD, "ratings_over_time")).orderBy("month").toPandas()
mensal["month"] = pd.to_datetime(mensal["month"])
colunas_pct = [c for c in mensal.columns if c.startswith("pct_")]
# Proporções anuais = média das proporções mensais, pesada pelo nº de avaliações de cada mês
anual = mensal.assign(year=mensal["month"].dt.year)
anual = (anual[colunas_pct].mul(anual["n_ratings"], axis=0).groupby(anual["year"]).sum()
         .div(anual.groupby("year")["n_ratings"].sum(), axis=0))

fig, eixos = plt.subplots(2, 1, figsize=(12, 8))
eixos[0].bar(mensal["month"], mensal["n_ratings"], width=25, color="#cccccc", label="avaliações por mês")
eixos[0].set_ylabel("Nº de avaliações")
ax2 = eixos[0].twinx()
ax2.plot(mensal["month"], mensal["avg_rating"].rolling(6, min_periods=1).mean(), color="#c44e52",
         label="média (móvel 6 meses)")
ax2.set_ylabel("Nota média", color="#c44e52")
ax2.grid(False)
eixos[0].set_title("Figura 14a. Volume mensal e nota média ao longo do tempo")
eixos[1].stackplot(anual.index, anual.T.values, labels=[c.replace("pct_", "").replace("_", ".") for c in colunas_pct],
                   colors=plt.cm.RdYlGn(np.linspace(0, 1, len(colunas_pct))))
eixos[1].set_ylim(0, 1)
eixos[1].xaxis.set_major_locator(MaxNLocator(integer=True))
eixos[1].set_ylabel("Proporção das avaliações")
eixos[1].set_title("Figura 14b. Distribuição das notas por ano")
eixos[1].legend(title="Nota", ncol=10, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.08))
fig.tight_layout()
plt.show()

In [0]:
# =============================================================================
# ETAPA 5.12.5 — Figura 15 (perguntas 6 e 7): utilizadores sinalizados, com e sem as primeiras 24 h
# =============================================================================
sens = spark.table(nome_tabela(GOLD, "flag_sensitivity")).toPandas()
x = np.arange(len(sens))
fig, ax = plt.subplots(figsize=(10, 4.5))
b1 = ax.bar(x - 0.2, sens["utilizadores_todas"], 0.4, label="todas as avaliações", color="#c44e52")
b2 = ax.bar(x + 0.2, sens["utilizadores_sem_24h"], 0.4, label="sem as primeiras 24 h", color="#4c72b0")
ax.bar_label(b1, padding=2, fontsize=8)
ax.bar_label(b2, padding=2, fontsize=8)
ax.set_xticks(x, sens["sinalizacao"], fontsize=9)
ax.set_ylabel("Nº de utilizadores sinalizados")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_title(f"Figura 15. Sinalização de utilizadores (total: {int(sens['total_utilizadores'].iloc[0]):,})")
ax.legend()
fig.tight_layout()
plt.show()
display(spark.table(nome_tabela(GOLD, "user_activity")).where("flag_any")
        .select("user_id", "n_ratings", "max_ratings_day", "max_same_second", "top_share",
                "flag_spike", "flag_same_second", "flag_concentration", "flag_any_sem_24h", "first_flag_date")
        .orderBy(F.desc("n_ratings")))
print("Uma queda sem as primeiras 24 h mostra sensibilidade ao período após a primeira avaliação observada; não identifica a data de registo.")

In [0]:
# =============================================================================
# ETAPA 5.12.6 — Figura 16 (perguntas 8 e 9): maiores subidas e descidas entre períodos
# =============================================================================
variacoes = spark.table(nome_tabela(GOLD, "movie_rating_shifts"))
extremos = (pd.concat([variacoes.orderBy(F.desc("variacao")).limit(8).toPandas(),
                       variacoes.orderBy("variacao").limit(8).toPandas()])
            .drop_duplicates("movie_id").sort_values("variacao"))
if len(extremos):
    rotulos = extremos["title"].str.slice(0, 35) + "  " + extremos["periodo_anterior"] + " → " + extremos["periodo_atual"]
    fig, ax = plt.subplots(figsize=(11, 0.4 * len(extremos) + 1.5))
    barras = ax.barh(rotulos, extremos["variacao"], color=np.where(extremos["variacao"] > 0, "#55a868", "#c44e52"))
    ax.bar_label(barras, labels=[f"n = {a} + {b}" for a, b in zip(extremos["n_anterior"], extremos["n_atual"])],
                 padding=3, fontsize=8)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlim(min(extremos["variacao"].min(), 0) - 0.8, max(extremos["variacao"].max(), 0) + 0.8)
    ax.set_xlabel("Variação da nota média entre períodos (pontos, escala 0.5-5)")
    ax.set_title(f"Figura 16. Maiores subidas e descidas (períodos de {ANOS_PERIODO} anos; n = avaliações em cada período)")
    fig.tight_layout()
    plt.show()
else:
    print("Nenhum filme tem avaliações suficientes em dois períodos; baixar MIN_AVALIACOES_PERIODO ou ANOS_PERIODO.")

In [0]:
# =============================================================================
# ETAPA 5.12.7 — Figura 17 (perguntas 10 e 11): concentrações no mesmo dia e participantes já sinalizados
# =============================================================================
anomalias = spark.table(nome_tabela(GOLD, "movie_day_anomalies")).orderBy(F.desc("z"))
display(anomalias.limit(20))
participantes = spark.table(nome_tabela(GOLD, "anomaly_participants")).toPandas()
por_tipo = anomalias.groupBy("tipo").count().toPandas()
n_anomalias = int(por_tipo["count"].sum()) if len(por_tipo) else 0
if len(participantes):
    ja_sinalizados = int(participantes["ja_sinalizado"].sum())
    pct = 100 * ja_sinalizados / len(participantes)
    fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
    eixos[0].bar(por_tipo["tipo"], por_tipo["count"],
                 color=["#55a868" if "altas" in t else "#c44e52" for t in por_tipo["tipo"]])
    eixos[0].set_ylabel("Nº de dias-filme anómalos")
    eixos[0].set_title(f"Figura 17a. {n_anomalias} concentrações anormais no mesmo dia")
    eixos[1].bar(["já sinalizados", "não sinalizados"], [ja_sinalizados, len(participantes) - ja_sinalizados],
                 color=["#c44e52", "#999999"])
    eixos[1].set_title(f"Figura 17b. Participantes: {pct:.1f}% já estavam sinalizados")
    eixos[1].set_ylabel("Nº de utilizadores")
    fig.tight_layout()
    plt.show()
    print(f"{len(participantes):,} utilizadores participaram em {n_anomalias} anomalias; {ja_sinalizados:,} ({pct:.1f}%) "
          "já tinham sido sinalizados por atividade invulgar antes da primeira participação.")
else:
    print("Não foram encontradas concentrações anormais com os limiares atuais.")

## 6. Deployment (CRISP-DM fase 6)

Num projeto real o pipeline não corre à mão: corre como um **Job** do Databricks, agendado ou disparado pela chegada de ficheiros.

**Como está feito neste projeto.** O notebook corre de cima para baixo, sem parâmetros de etapa, por isso o Job mais simples tem uma só tarefa:

1. Menu da esquerda **Jobs & Pipelines** → **Create** → **Job**.
2. *Task name* `pipeline_filmes`, *Type* **Notebook**, *Path* = este notebook, *Compute* **Serverless**.
3. *Trigger*: para este notebook, usar **Run now** depois de a batch estar completa. Um *File arrival* inicia o notebook inteiro: o Auto Loader acrescenta ficheiros novos à bronze, mas a silver e a gold são reconstruídas e a secção seguinte também corre. Se automatizarem mais tarde, separar primeiro a ingestão da análise e monitorizar apenas as pastas de entrada. Não monitorizar a raiz `raw`, que inclui `_checkpoints`.

**Como ficaria com mais volume de dados.** Cada etapa passaria a ser um notebook próprio e uma tarefa do Job. As três fontes são independentes, por isso correriam em paralelo, e o Databricks desenharia o grafo e mostraria o estado de cada tarefa:

```
   bronze + silver da fonte complementar ┐
   bronze + silver IMDb ──────┼──► silver integrada ──► gold ──► análise e dashboard
   bronze + silver MovieLens ─┘
```

Optámos por um só notebook porque a entrega é um notebook único. Como a execução final usa MovieLens 32m, medir o tempo e os recursos antes de automatizar execuções frequentes.

### 6.1 Consultas SQL guardadas e dashboard (Databricks SQL)

Os gráficos deste notebook servem a análise. Para quem consome os resultados (gestão, outras equipas) o habitual é um **dashboard** alimentado por **consultas SQL guardadas**, que leem só a camada gold.

| # | Consulta | Responde a | Tabela gold |
|---|---|---|---|
| Q1 | Nota média por fonte e década | H1, H5 | `ratings_by_decade` |
| Q2 | Diferença IMDb − MovieLens por género | H2 | `gap_by_genre` |
| Q3 | Diferença por idioma original | H3 | `gap_by_language` |
| Q4 | Consenso por escalão de orçamento | H4 | `gap_by_budget` |
| Q5 | Filmes onde as fontes mais discordam | H1 | `movie_rating_gaps` |
| Q6 | Nota média por fonte e década, no star schema | 4.4 | `fact_rating` + dimensões |
| Q7 | Evolução mensal das avaliações | 4.3, pergunta 5 | `ratings_over_time` |
| Q8 | Utilizadores sinalizados, com e sem as primeiras 24 h | 4.3, perguntas 6-7 | `flag_sensitivity` |
| Q9 | Qualidade dos dados na última execução | secção 3.5 | `dq_results` |

**O que as células seguintes fazem:**
1. Correm as 7 consultas atualmente definidas e mostram o resultado; Q3 e Q4 podem ser incluídas quando o respetivo código e tabelas forem ativados.
2. Com o widget `publicar_sql = sim`, guardam-nas em **Queries** e criam (ou atualizam) o dashboard **Filmes: as notas dependem de quem as dá?** em **Dashboards**, na pasta `abd_project_sql` do vosso utilizador. Antes de criar, verificam se já existem com o mesmo nome: se sim, atualizam em vez de duplicar.

Se a publicação automática falhar (permissões, versão da biblioteca), o notebook avisa e continua. Nesse caso, as consultas podem ser criadas à mão em **SQL Editor**, copiando o texto mostrado pela célula seguinte, e o dashboard em **Dashboards → Create dashboard**, usando essas consultas.

In [0]:
# =============================================================================
# ETAPA 6.1 — Consultas SQL sobre a gold
# =============================================================================
def gold_tbl(tabela):
    return nome_tabela(GOLD, tabela)


SQL_QUERIES = {
    "Q1 - Nota média por fonte e década": f"""
        SELECT decade, n_movies, avg_imdb, avg_ml
        FROM {gold_tbl('ratings_by_decade')}
        ORDER BY decade""",
    "Q2 - Diferença IMDb-MovieLens por género": f"""
        SELECT genre, n_movies, avg_gap, sd_gap, avg_imdb, avg_ml
        FROM {gold_tbl('gap_by_genre')}
        ORDER BY avg_gap DESC""",
    # [TMDB desativada] Q3 e Q4 dependem da TMDB (e Q1 deixou de ter avg_tmdb)
    # "Q3 - Diferença por idioma original": f"""
        # SELECT original_language, n_movies, avg_gap_ml, avg_gap_tmdb
        # FROM {gold_tbl('gap_by_language')}
        # ORDER BY n_movies DESC
        # LIMIT 15""",
    # "Q4 - Consenso por escalão de orçamento": f"""
        # SELECT budget_band, n_movies, avg_gap, avg_abs_gap
        # FROM {gold_tbl('gap_by_budget')}
        # ORDER BY budget_band""",
    "Q5 - Filmes onde as fontes mais discordam": f"""
        SELECT title, start_year, imdb_rating, ml_rating_10, gap_imdb_ml, imdb_votes, ml_num_ratings
        FROM {gold_tbl('movie_rating_gaps')}
        ORDER BY abs_gap DESC
        LIMIT 20""",
    "Q6 - Star schema: nota média por fonte e década": f"""
        SELECT y.decade, s.source_name, ROUND(AVG(f.rating_10), 2) AS nota_media, COUNT(*) AS n
        FROM {gold_tbl('fact_rating')} f
        JOIN {gold_tbl('dim_source')} s ON f.source_key = s.source_key
        JOIN {gold_tbl('dim_year')}   y ON f.year_key   = y.year_key
        WHERE y.year_key <> -1
        GROUP BY y.decade, s.source_name
        ORDER BY y.decade, s.source_name""",
    "Q7 - Evolução mensal das avaliações": f"""
        SELECT month, n_ratings, avg_rating, n_users
        FROM {gold_tbl('ratings_over_time')}
        ORDER BY month""",
    "Q8 - Utilizadores sinalizados com e sem 24h": f"""
        SELECT sinalizacao, utilizadores_todas, utilizadores_sem_24h, variacao_pct
        FROM {gold_tbl('flag_sensitivity')}""",
    "Q9 - Qualidade dos dados na última execução": f"""
        SELECT tabela, regra, linhas, violacoes, pct_violacoes, _checked_at
        FROM {gold_tbl('dq_results')}
        WHERE _batch_id = (SELECT MAX(_batch_id) FROM {gold_tbl('dq_results')})
        ORDER BY pct_violacoes DESC""",
}

for nome_consulta, texto_sql in SQL_QUERIES.items():
    print(nome_consulta)
    display(spark.sql(texto_sql))

In [0]:
# [Auxiliar] Publicação das consultas e do dashboard no Databricks SQL, através do Databricks SDK
# (biblioteca Python oficial, já instalada no Databricks e autenticada com o vosso utilizador).
# O que interessa perceber: cada widget do dashboard = uma consulta + um tipo de gráfico + as colunas dos eixos.
import json

DASHBOARD_NAME = "Filmes: as notas dependem de quem as dá?"

# (consulta, tipo de gráfico, coluna do eixo X, colunas do eixo Y, posição x, posição y, largura, altura)
# A grelha do dashboard tem 6 colunas de largura.
DASHBOARD_WIDGETS = [
    ("Q1 - Nota média por fonte e década",         "line",  "decade",            ["avg_imdb", "avg_ml"], 0, 0, 3, 6),   # [TMDB desativada] "avg_tmdb"
    ("Q2 - Diferença IMDb-MovieLens por género",   "bar",   "genre",             ["avg_gap"],                        3, 0, 3, 6),
    # [TMDB desativada] 
    # ("Q3 - Diferença por idioma original",         "bar",   "original_language", ["avg_gap_ml"],                     0, 6, 3, 6),
    # ("Q4 - Consenso por escalão de orçamento",     "bar",   "budget_band",       ["avg_abs_gap"],                    3, 6, 3, 6),
    ("Q7 - Evolução mensal das avaliações",        "line",  "month",             ["n_ratings"],                      0, 12, 3, 6),
    ("Q8 - Utilizadores sinalizados com e sem 24h", "bar",  "sinalizacao",       ["utilizadores_todas", "utilizadores_sem_24h"], 3, 12, 3, 6),
    ("Q5 - Filmes onde as fontes mais discordam",  "table", None, ["title", "start_year", "imdb_rating", "ml_rating_10", "gap_imdb_ml"], 0, 18, 6, 7),
    ("Q9 - Qualidade dos dados na última execução", "table", None, ["tabela", "regra", "linhas", "violacoes", "pct_violacoes"], 0, 25, 6, 6),
]


def dashboard_json():
    """Constrói a definição do dashboard (formato JSON dos AI/BI Dashboards)."""
    ids = {name: f"q{i + 1}" for i, name in enumerate(SQL_QUERIES)}
    datasets = [{"name": ids[n], "displayName": n, "queryLines": [" ".join(q.split())]} for n, q in SQL_QUERIES.items()]
    layout = []
    for i, (qname, kind, x_col, y_cols, px, py, w, h) in enumerate(DASHBOARD_WIDGETS):
        cols = ([x_col] if x_col else []) + y_cols
        query = {"name": "main_query",
                 "query": {"datasetName": ids[qname], "disaggregated": True,
                           "fields": [{"name": c, "expression": f"`{c}`"} for c in cols]}}
        if kind == "table":
            spec = {"version": 1, "widgetType": "table",
                    "encodings": {"columns": [{"fieldName": c, "displayName": c} for c in y_cols]}}
        else:
            x_scale = "temporal" if x_col == "month" else ("quantitative" if x_col == "decade" else "categorical")
            spec = {"version": 3, "widgetType": kind,
                    "encodings": {"x": {"fieldName": x_col, "scale": {"type": x_scale}, "displayName": x_col},
                                  "y": {"fields": [{"fieldName": c, "displayName": c} for c in y_cols],
                                        "scale": {"type": "quantitative"}}}}
        spec["frame"] = {"showTitle": True, "title": qname}
        layout.append({"widget": {"name": f"w{i + 1}", "queries": [query], "spec": spec},
                       "position": {"x": px, "y": py, "width": w, "height": h}})
    return json.dumps({"datasets": datasets,
                       "pages": [{"name": "principal", "displayName": "Resultados", "layout": layout}]})


def publicar_sql():
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service import sql as dbsql
    w = WorkspaceClient()

    # 1) Warehouse SQL (na Free Edition há um: "Serverless Starter Warehouse")
    warehouses = list(w.warehouses.list())
    if not warehouses:
        raise RuntimeError("não há nenhum SQL Warehouse disponível")
    wh = next((x for x in warehouses if "starter" in (x.name or "").lower()), warehouses[0])

    # 2) Pasta de destino no workspace do utilizador
    folder = f"/Users/{w.current_user.me().user_name}/abd_project_sql"
    w.workspace.mkdirs(folder)

    # 3) Consultas guardadas: verificar se já existe uma com o mesmo nome; se sim atualizar, senão criar
    existing = {q.display_name: q.id for q in w.queries.list()}
    for name, sql_text in SQL_QUERIES.items():
        text = "\n".join(line.strip() for line in sql_text.strip().splitlines())
        if name in existing:
            w.queries.update(existing[name], update_mask="query_text,warehouse_id",
                             query=dbsql.UpdateQueryRequestQuery(query_text=text, warehouse_id=wh.id))
            print(f"[sql] consulta atualizada: {name}")
        else:
            w.queries.create(query=dbsql.CreateQueryRequestQuery(display_name=name, query_text=text,
                                                                 warehouse_id=wh.id, parent_path=folder))
            print(f"[sql] consulta criada:     {name}")

    # 4) Dashboard: o mesmo princípio (atualiza se já existir com este nome)
    from databricks.sdk.service.dashboards import Dashboard
    definition = dashboard_json()
    found = next((d for d in w.lakeview.list() if d.display_name == DASHBOARD_NAME), None)
    fields = dict(display_name=DASHBOARD_NAME, serialized_dashboard=definition, warehouse_id=wh.id)
    try:   # versões recentes do SDK recebem um objeto Dashboard; as antigas recebem os campos diretamente
        if found:
            dash = w.lakeview.update(found.dashboard_id, dashboard=Dashboard(**fields))
        else:
            dash = w.lakeview.create(dashboard=Dashboard(parent_path=folder, **fields))
    except TypeError:
        if found:
            dash = w.lakeview.update(found.dashboard_id, **fields)
        else:
            dash = w.lakeview.create(parent_path=folder, **fields)
    print(f"[sql] dashboard {'atualizado' if found else 'criado'}: {DASHBOARD_NAME}")
    w.lakeview.publish(dash.dashboard_id, warehouse_id=wh.id, embed_credentials=True)
    print(f"[sql] dashboard publicado. Abrir em Dashboards ou em {folder}")


if PUBLICAR_SQL:
    try:
        publicar_sql()
    except Exception as e:
        print(f"[aviso] publicação automática não concluída ({type(e).__name__}: {str(e)[:200]}).\n"
              "        Os resultados acima não são afetados. Criar à mão seguindo as instruções da secção 6.1.")

### 6.2 Inventário final e estado do pipeline

In [0]:
# =============================================================================
# ETAPA 6.2 — Inventário final: nº de linhas de cada tabela (contado uma única vez) e diagrama do pipeline
# =============================================================================
# 1) Contar as linhas de cada tabela, uma única vez, no fim da execução
inventario = {}
for camada in CAMADAS:
    for linha in spark.sql(f"SHOW TABLES IN {CATALOGO}.{camada}").collect():
        if not linha.isTemporary:
            inventario[nome_tabela(camada, linha.tableName)] = spark.table(nome_tabela(camada, linha.tableName)).count()
display(spark.createDataFrame([(nome.split(".")[1], nome.split(".")[2], n) for nome, n in inventario.items()],
                              "camada string, tabela string, linhas long").orderBy("camada", "tabela"))


# 2) Diagrama do pipeline (SVG), pintado com as contagens acima: verde = todas as tabelas criadas,
#    amarelo = parcial, cinzento = por criar. Não é preciso perceber o código de desenho.
def mostrar_pipeline(contagens):
    cores = {"completo": ("#d4edda", "#2e7d32"), "parcial": ("#fff3cd", "#b8860b"), "vazio": ("#eeeeee", "#9e9e9e")}

    def caixa(x, y, titulo, tabelas, w=180, h=64):
        existentes = [t for t in tabelas if t in contagens]
        linhas = sum(contagens[t] for t in existentes)
        estado = "completo" if len(existentes) == len(tabelas) else ("parcial" if existentes else "vazio")
        fundo, borda = cores[estado]
        return (f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="8" fill="{fundo}" stroke="{borda}" stroke-width="2"/>'
                f'<text x="{x + w / 2}" y="{y + 24}" text-anchor="middle" font-weight="bold" font-size="13">{titulo}</text>'
                f'<text x="{x + w / 2}" y="{y + 44}" text-anchor="middle" font-size="11">'
                f'{len(existentes)}/{len(tabelas)} tabelas · {linhas:,} linhas</text>')

    def seta(x1, y1, x2, y2):
        return f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="#555" stroke-width="1.6" marker-end="url(#seta)"/>'

    linhas_y = {"imdb": 140, "movielens": 240}            # [TMDB desativada] "tmdb": 40
    rotulos = {"imdb": "IMDb (TSV)", "movielens": "MovieLens (CSV)"}   # [TMDB desativada] "tmdb": "TMDB (JSON)"
    partes = []
    for fonte, y in linhas_y.items():
        partes.append(f'<rect x="10" y="{y}" width="150" height="64" rx="8" fill="{cores["completo"][0]}" '
                      f'stroke="{cores["completo"][1]}" stroke-width="2"/>'
                      f'<text x="85" y="{y + 36}" text-anchor="middle" font-weight="bold" font-size="13">{rotulos[fonte]}</text>')
        partes.append(caixa(200, y, f"Bronze · {fonte}", [nome_tabela(BRONZE, t) for t in TABELAS_BRONZE[fonte]]))
        partes.append(caixa(420, y, f"Silver · {fonte}", [nome_tabela(SILVER, t) for t in TABELAS_SILVER[fonte]]))
        partes += [seta(160, y + 32, 198, y + 32), seta(380, y + 32, 418, y + 32), seta(600, y + 32, 648, 172)]
    partes.append(caixa(650, 140, "Silver integrada", [nome_tabela(SILVER, "movies_unified")]))
    partes.append(seta(830, 172, 868, 172))
    partes.append(caixa(870, 140, "Gold", [nome_tabela(GOLD, t) for t in TABELAS_GOLD + TABELAS_STAR]))
    for x, texto in [(85, "Fontes"), (290, "Bronze"), (510, "Silver"), (740, "Integração"), (960, "Gold")]:
        partes.append(f'<text x="{x}" y="25" text-anchor="middle" font-size="12" fill="#555">{texto}</text>')
    legenda = ('<text x="10" y="335" font-size="11" fill="#333">'
               '<tspan fill="#2e7d32">■</tspan> criado · <tspan fill="#b8860b">■</tspan> parcial · '
               '<tspan fill="#9e9e9e">■</tspan> por criar</text>')
    displayHTML(f"""
    <div style="background:#fff;padding:8px;font-family:sans-serif;color:#222">
      <h3 style="margin:4px 0">Estado do pipeline no fim da execução <span style="font-weight:normal;font-size:13px">(lote: {LOTE})</span></h3>
      <svg width="1060" height="345" xmlns="http://www.w3.org/2000/svg" style="font-family:sans-serif">
        <defs><marker id="seta" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
          <path d="M0,0 L8,4 L0,8 z" fill="#555"/></marker></defs>
        {''.join(partes)}{legenda}
      </svg>
    </div>""")


mostrar_pipeline(inventario)

## 7. Conclusões

**Resposta à pergunta de negócio:** nos filmes comparáveis, as notas das fontes estão fortemente relacionadas, mas há uma diferença média pequena e consistente.

### Resultados das hipóteses

* **H1 (as fontes diferem):** nos mesmos **26 614 filmes**, a média do IMDb é **6,44** e a do MovieLens, convertido para a escala 1–10, é **6,33**. O gap médio IMDb − MovieLens é **+0,11 pontos** (IC 95% aproximado: **+0,10 a +0,11**); a correlação é **r = 0,87**. Há evidência de diferença média nesta amostra, embora o efeito médio seja pequeno. (Fig. 3)
* **H2 (género):** é parcialmente apoiada, mas não provada: os maiores gaps médios surgem em Animation (+0,320), Film-Noir (+0,314), News (+0,298; n=22, estimativa incerta) e Documentary (+0,277); Sci-Fi, Horror e Mystery ficam perto de zero. Na regressão, Horror, Thriller e Romance associam-se a um gap menor (p < 0,001). A escala comum permite comparar as notas, mas não prova independência; com R² = 0,037 e sem correção para múltiplos testes, os resultados são exploratórios.
* **H3 (idioma) e H4 (orçamento):** não avaliadas nesta execução.
* **H5 (década):** a nota IMDb média é **6,83** antes de 1980 e **6,32** desde 2000. Na regressão, o gap IMDb − MovieLens diminui cerca de **0,033 pontos por década**. O padrão é compatível com H5, mas não demonstra um efeito de sobrevivência nem uma causa. (Fig. 4)
* **Texto livre:** nos filmes com gaps absolutos de pelo menos 1 ponto, surgem tags como bd-r e murder quando o IMDb é mais alto, e woman director e independent film quando o MovieLens é mais alto. São descrições criadas por utilizadores, não explicações do gap. (Fig. 8)

### Utilizadores e sinalizações exploratórias

O conjunto tem **200 948 utilizadores**; a mediana é 73 avaliações e os 10% mais ativos fazem 47% do total. As regras sinalizam pelo menos um comportamento em **88 604 utilizadores (44,1%)**; ao excluir as primeiras 24 horas após a primeira avaliação observada, o total cai para **26 156 (−70,5%)**. As sinalizações de avaliações no mesmo segundo passam de **73 745 para 17 504 (−76,3%)**. Foram também sinalizados **6 539 pares filme-dia**, com 32 001 participantes; 4 768 participantes (14,9%) já tinham uma sinalização anterior. Estes resultados dependem de heurísticas e não demonstram fraude ou coordenação. A primeira avaliação observada não identifica a data de registo do utilizador.

### Implicações e limitações

Uma média conjunta das duas notas na escala 1–10 pode servir como resumo geral, mantendo as notas por fonte acessíveis. Mostrar obrigatoriamente cada fonte quando o gap ultrapassa 1 ponto é uma regra de produto a validar: esta execução não calcula que proporção dos filmes excede esse limiar.

A análise usa apenas filmes com correspondência entre as fontes; os utilizadores do MovieLens não representam o público geral. Na silver, 14,98% das linhas de imdb_movies não têm ano de estreia e 3,71% dos filmes em movies_unified não têm nota MovieLens. A regressão explica **3,7%** da variação do gap; os coeficientes mostram associações, não causalidade.

### Processo e tecnologia

A execução final processou **32 000 204 avaliações MovieLens** e **12 963 111 linhas de title.basics do IMDb** (que incluem vários tipos de título, não apenas filmes). O plano físico mostra Photon e broadcast joins. Sete consultas SQL e o dashboard foram atualizados na conta desta execução. Não foram medidos o tempo/custo do run nem o comportamento de um Job agendado ou noutra conta; por isso, a execução frequente e a portabilidade ainda precisam de validação.